In [ ]:
# In[1]:

import pandas as pd
import pytz

# Set timezone variable for UTC+8 as required (not applied to numeric timestamps)
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into DataFrames (reuse variable names as requested)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to compute compact summary for a DataFrame
def compact_summary(df, name):
    rows = df.shape[0]
    min_ts = df['timestamp'].min() if 'timestamp' in df.columns else pd.NA
    max_ts = df['timestamp'].max() if 'timestamp' in df.columns else pd.NA
    return pd.DataFrame({
        'file': [name],
        'rows': [rows],
        'min_timestamp': [min_ts],
        'max_timestamp': [max_ts]
    })

# Create summary for each file and combine
summary_metric = compact_summary(metric_df, 'metric.csv')
summary_trace = compact_summary(trace_df, 'trace.csv')
summary_log = compact_summary(log_df, 'log.csv')
summary_error = compact_summary(error_df, 'error_logs.csv')

summary_df = pd.concat([summary_metric, summary_trace, summary_log, summary_error], ignore_index=True)

# Unique cmdb_id values across all files (deduplicated and sorted)
cmdb_series_list = []
for df in (metric_df, trace_df, log_df, error_df):
    if 'cmdb_id' in df.columns:
        cmdb_series_list.append(df['cmdb_id'].dropna().astype(str))
if cmdb_series_list:
    unique_cmdb = pd.concat(cmdb_series_list).drop_duplicates().sort_values().reset_index(drop=True)
else:
    unique_cmdb = pd.Series(dtype=str)

unique_cmdb_df = pd.DataFrame({'cmdb_id': unique_cmdb})

# Top 50 kpi_name from metric_df
if 'kpi_name' in metric_df.columns:
    top_kpis_df = metric_df['kpi_name'].value_counts().head(50).reset_index()
    top_kpis_df.columns = ['kpi_name', 'count']
else:
    top_kpis_df = pd.DataFrame(columns=['kpi_name', 'count'])

# Top 50 trace_name from trace_df
if 'trace_name' in trace_df.columns:
    top_traces_df = trace_df['trace_name'].value_counts().head(50).reset_index()
    top_traces_df.columns = ['trace_name', 'count']
else:
    top_traces_df = pd.DataFrame(columns=['trace_name', 'count'])

# Top 50 log_name from log_df
if 'log_name' in log_df.columns:
    top_logs_df = log_df['log_name'].value_counts().head(50).reset_index()
    top_logs_df.columns = ['log_name', 'count']
else:
    top_logs_df = pd.DataFrame(columns=['log_name', 'count'])

# Display compact results (as requested, use variable names; multiple variables separated by commas)
summary_df, unique_cmdb_df, top_kpis_df, top_traces_df, top_logs_df

```
Out[1]:
```
```python
# Compose a plain-English summary string based on previously loaded results
# Reuse kernel variables: summary_df, unique_cmdb_df, top_kpis_df, top_traces_df, top_logs_df

# Helper to format file summary rows
file_lines = []
for _, r in summary_df.iterrows():
    file_lines.append(f"{r['file']}: rows={int(r['rows'])}, min_timestamp={r['min_timestamp']}, max_timestamp={r['max_timestamp']}")

# Unique cmdb ids
cmdb_list = unique_cmdb_df['cmdb_id'].tolist()
cmdb_count = len(cmdb_list)
cmdb_display = ", ".join(cmdb_list)

# Helper to get top N examples from a counts DF
def top_examples(df, name_col, count_col, n=3):
    if df.shape[0] == 0:
        return "none"
    n = min(n, df.shape[0])
    return "; ".join([f"{df.iloc[i][name_col]} ({int(df.iloc[i][count_col])})" for i in range(n)])

metric_examples = top_examples(top_kpis_df, 'kpi_name', 'count', 3)
trace_examples = top_examples(top_traces_df, 'trace_name', 'count', 3)
log_examples = top_examples(top_logs_df, 'log_name', 'count', 3)

metric_count = top_kpis_df.shape[0]
trace_count = top_traces_df.shape[0]
log_count = top_logs_df.shape[0]

summary_text = (
    "Summary of loaded telemetry files:\n\n"
    "- File summaries:\n  " + "\n  ".join(file_lines) + "\n\n"
    f"- Unique cmdb_id: {cmdb_count} entries. List: {cmdb_display}\n\n"
    f"- metric.csv: top {metric_count} kpi_name values (examples: {metric_examples}).\n"
    f"- trace.csv: top {trace_count} trace_name values (examples: {trace_examples}).\n"
    f"- log.csv: top {log_count} log_name values (examples: {log_examples}).\n\n"
    "Notes: error_logs.csv contains 0 rows. Full top-50 lists are available in variables "
    "top_kpis_df, top_traces_df, and top_logs_df."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows  min_timestamp  max_timestamp
0      metric.csv  27415   1.615514e+09   1.615516e+09
1       trace.csv   6894   1.615514e+09   1.615516e+09
2         log.csv    480   1.615514e+09   1.615516e+09
3  error_logs.csv      0            NaN            NaN,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
24  apache02
25  dockerA1
26  dockerA2
27  dockerB1
28  dockerB2

[29 rows x 1 columns],                                              kpi_name  count
0        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    363
1      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    359
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    356
3       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    356
4                  OSLinux-OSLinux_ZABBIX_Host_Uptime    356
..                                                ...    ...
45  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...    175
46  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...    175
47    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent    175
48  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...    175
49  OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPe...    175

[50 rows x 2 columns],                            trace_name  count
0            trace.self.duration_mean    360
1             trace.self.duration_p95    360
2                trace.self.row_count    360
3          trace.to_MG01.duration_p95    240
4             trace.to_MG01.row_count    240
..                                ...    ...
45   trace.from_dockerB1.duration_p95     60
46  trace.from_dockerB1.duration_mean     60
47      trace.from_dockerA2.row_count     60
48   trace.from_dockerA2.duration_p95     60
49  trace.from_dockerA2.duration_mean     60

[50 rows x 2 columns],            log_name  count
0   log.error_count    187
1     log.row_count    187
2  log.gc_pause_max    106)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing metric_df from kernel

# 1) Restrict metric_df to allowed components
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
metric_restricted_df = metric_df[metric_df['cmdb_id'].isin(allowed_components)].copy()

# 2) Compute global p95 and p05 per component-kpi across the entire restricted series
# (do NOT filter by time for thresholds)
grp = metric_restricted_df.groupby(['cmdb_id', 'kpi_name'])['value']
thresholds_metric_df = grp.quantile(0.95).reset_index().rename(columns={'value':'p95'})
p05_df = grp.quantile(0.05).reset_index().rename(columns={'value':'p05'})
thresholds_metric_df = thresholds_metric_df.merge(p05_df, on=['cmdb_id','kpi_name'])

# 3) Filter metric_df to the incident time window: March 12, 2021 10:00 to 10:30 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2021,3,12,10,0,0))
end_dt   = tz.localize(datetime(2021,3,12,10,30,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# Use the restricted set when filtering window (we were instructed to restrict first)
metric_window_df = metric_restricted_df[(metric_restricted_df['timestamp'] >= start_ts) &
                                       (metric_restricted_df['timestamp'] <= end_ts)].copy()

# 4) Join thresholds into the windowed rows so we can compare against p95/p05
window_with_thresh_df = metric_window_df.merge(thresholds_metric_df, on=['cmdb_id','kpi_name'], how='left')

# Prepare helper columns for comparisons
window_with_thresh_df['over_p95'] = window_with_thresh_df['value'] > window_with_thresh_df['p95']
window_with_thresh_df['below_p05'] = window_with_thresh_df['value'] < window_with_thresh_df['p05']

# Aggregate required stats per component-kpi in the window
def agg_group(df):
    count_points = int(df.shape[0])
    count_over = int(df['over_p95'].sum())
    count_below = int(df['below_p05'].sum())
    earliest_over_ts = int(df.loc[df['over_p95'], 'timestamp'].min()) if count_over>0 else pd.NA
    earliest_below_ts = int(df.loc[df['below_p05'], 'timestamp'].min()) if count_below>0 else pd.NA
    max_value = float(df['value'].max()) if count_points>0 else pd.NA
    min_value = float(df['value'].min()) if count_points>0 else pd.NA
    mean_value = float(df['value'].mean()) if count_points>0 else pd.NA
    return pd.Series({
        'count_points_in_window': count_points,
        'count_over_p95': count_over,
        'earliest_over_p95_timestamp': earliest_over_ts,
        'max_value_in_window': max_value,
        'count_below_p05': count_below,
        'earliest_below_p05_timestamp': earliest_below_ts,
        'min_value_in_window': min_value,
        'mean_value_in_window': mean_value
    })

agg_df = window_with_thresh_df.groupby(['cmdb_id','kpi_name']).apply(agg_group).reset_index()

# 4b) Join aggregated window stats with thresholds_metric_df to include p95/p05
joined_df = agg_df.merge(thresholds_metric_df, on=['cmdb_id','kpi_name'], how='left')

# 5) Prepare compact table sorted by count_over_p95 desc, then count_below_p05 desc (as secondary if over count zeros)
joined_df['sort_second'] = joined_df['count_below_p05']  # secondary sort key
final_df = joined_df.sort_values(by=['count_over_p95','sort_second'], ascending=[False, False])

# Select requested columns and limit to top 20 rows
final_columns = ['cmdb_id','kpi_name','p95','p05',
                 'count_points_in_window','count_over_p95','earliest_over_p95_timestamp',
                 'max_value_in_window','count_below_p05','earliest_below_p05_timestamp',
                 'min_value_in_window','mean_value_in_window']
final_top20_df = final_df[final_columns].head(20).reset_index(drop=True)

# 6) Counts: total distinct component-kpi series in thresholds_metric_df, and number with at least one anomaly in window
total_series_count = int(thresholds_metric_df.shape[0])
# Number of series that had at least one anomaly in the window (count_over_p95>0 or count_below_p05>0)
anomaly_series_count = int(((final_df['count_over_p95']>0) | (final_df['count_below_p05']>0)).sum())

# Return compact results (as variables)
final_top20_df, total_series_count, anomaly_series_count

```
Out[2]:
```
```python
# Compose a concise plain-English summary based on existing kernel results
# Reuse variables: final_top20_df, total_series_count, anomaly_series_count

window_start_ts = 1615514400  # March 12, 2021 10:00 UTC+8
window_end_ts   = 1615516200  # March 12, 2021 10:30 UTC+8

# Prepare top examples (up to 5) for compact display
examples = []
for _, r in final_top20_df.head(5).iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    pts = int(r['count_points_in_window']) if not pd.isna(r['count_points_in_window']) else 0
    over = int(r['count_over_p95']) if not pd.isna(r['count_over_p95']) else 0
    below = int(r['count_below_p05']) if not pd.isna(r['count_below_p05']) else 0
    e_over = int(r['earliest_over_p95_timestamp']) if not pd.isna(r['earliest_over_p95_timestamp']) else None
    e_below = int(r['earliest_below_p05_timestamp']) if not pd.isna(r['earliest_below_p05_timestamp']) else None
    maxv = float(r['max_value_in_window']) if not pd.isna(r['max_value_in_window']) else None
    examples.append(f"{cmdb} | {kpi} | points={pts} | over_p95={over} | below_p05={below} | earliest_over={e_over} | earliest_below={e_below} | max={maxv}")

pct_anomalous = round(100 * anomaly_series_count / total_series_count, 2) if total_series_count>0 else 0.0

summary_text = (
    f"Summary of anomaly scan (metric data restricted to allowed components):\n\n"
    f"- Incident window: March 12, 2021 10:00 to 10:30 (UTC+8) -> timestamps {window_start_ts} to {window_end_ts}.\n"
    f"- Thresholds (global P95/P05) were computed per component-kpi using the entire available series (no time filtering).\n"
    f"- Total distinct component-kpi series evaluated: {total_series_count}.\n"
    f"- Series with at least one anomaly in window (value>p95 or value<p05): {anomaly_series_count} ({pct_anomalous}%).\n\n"
    f"- Top examples (up to 5) from the top-20 sorted by count_over_p95:\n  " + "\n  ".join(examples) + "\n\n"
    f"Notes: Most top rows are IG01/IG02 and MG/Mysql CPU or JVM Memory KPIs. Each example shows: cmdb_id | kpi_name | points in window | count_over_p95 | count_below_p05 | earliest_over_p95_timestamp | earliest_below_p05_timestamp | max_value_in_window.\n"
    f"Full compact table of top-20 rows is available in variable final_top20_df."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                   kpi_name           p95           p05  count_points_in_window  count_over_p95 earliest_over_p95_timestamp  max_value_in_window  count_below_p05 earliest_below_p05_timestamp  min_value_in_window  mean_value_in_window
0      IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.875126e+08  1.077591e+08                    31.0             2.0                1615514640.0         8.652382e+08              2.0                 1615514940.0         7.262545e+07          4.607712e+08
1      IG01                    OSLinux-CPU_CPU_CPULoad  1.206500e+00  1.004500e+00                    30.0             2.0                1615514880.0         1.340000e+00              2.0                 1615515600.0         1.000000e+00          1.081000e+00
2      IG01                 OSLinux-CPU_CPU_CPUSysTime  5.502473e-01  3.442689e-01                    30.0             2.0                1615514940.0         5.697180e-01              2.0                 1615514520.0         3.161000e-01          4.506746e-01
3      IG01                OSLinux-CPU_CPU_CPUUserTime  2.541316e+01  2.528865e+01                    30.0             2.0                1615514820.0         2.542937e+01              2.0                 1615514520.0         2.527235e+01          2.535489e+01
4      IG01                     OSLinux-CPU_CPU_CPUWio  2.427254e+00  2.035795e-02                    30.0             2.0                1615516080.0         2.790000e+00              2.0                 1615514520.0         1.660000e-02          4.135154e-01
5      IG01                OSLinux-CPU_CPU_CPUidleutil  7.402932e+01  7.155711e+01                    30.0             2.0                1615514520.0         7.414540e+01              2.0                 1615516080.0         7.132220e+01          7.352972e+01
6      IG01         OSLinux-OSLinux_ZABBIX_Host_Uptime  6.715161e+06  6.713538e+06                    31.0             2.0                1615516140.0         6.715251e+06              2.0                 1615514400.0         6.713447e+06          6.714349e+06
7      IG02  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.647473e+08  1.165967e+08                    31.0             2.0                1615514400.0         9.010234e+08              2.0                 1615514580.0         9.158998e+07          4.822746e+08
8      IG02                 OSLinux-CPU_CPU_CPUSysTime  6.862123e-01  4.293150e-01                    30.0             2.0                1615515900.0         8.288000e-01              2.0                 1615515660.0         4.199000e-01          5.421080e-01
9      IG02                OSLinux-CPU_CPU_CPUUserTime  7.725361e+01  2.539025e+01                    30.0             2.0                1615516080.0         8.679655e+01              2.0                 1615514520.0         2.536786e+01          3.222647e+01
10     IG02                     OSLinux-CPU_CPU_CPUWio  1.620916e+00  3.928645e-02                    30.0             2.0                1615515840.0         3.633500e+00              2.0                 1615515180.0         2.494900e-02          4.206997e-01
11     IG02                OSLinux-CPU_CPU_CPUidleutil  7.375016e+01  2.093193e+01                    30.0             2.0                1615514520.0         7.379120e+01              2.0                 1615516080.0         1.185599e+01          6.648860e+01
12     IG02         OSLinux-OSLinux_ZABBIX_Host_Uptime  6.715160e+06  6.713577e+06                    30.0             2.0                1615516140.0         6.715248e+06              2.0                 1615514400.0         6.713490e+06          6.714366e+06
13     MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.943934e+08  2.622535e+08                    31.0             2.0                1615514520.0         1.023737e+09              2.0                 1615514400.0         2.327843e+08          5.945870e+08
14     MG02  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.194533e+09  4.317653e+08                    31.0             2.0                1615514520.0         1.213109e+09              2.0                 1615515420.0         4.306024e+08          8.089397e+08
15  Mysql01                 OSLinux-CPU_CPU_CPUSysTime  4.928500e-01  3.596880e-01                    31.0             2.0                1615514940.0         5.320030e-01              2.0                 1615515360.0         3.535000e-01          4.218862e-01
16  Mysql01                OSLinux-CPU_CPU_CPUUserTime  5.472505e-01  3.846550e-01                    31.0             2.0                1615515120.0         6.651150e-01              2.0                 1615515720.0         3.493000e-01          4.554163e-01
17  Mysql01                     OSLinux-CPU_CPU_CPUWio  1.416635e+01  1.031770e+00                    31.0             2.0                1615516140.0         1.536510e+01              2.0                 1615515300.0         6.443000e-01          4.694583e+00
18  Mysql01                OSLinux-CPU_CPU_CPUidleutil  9.803688e+01  8.504770e+01                    31.0             2.0                1615515300.0         9.848270e+01              2.0                 1615516140.0         8.367020e+01          9.440558e+01
19  Mysql01         OSLinux-OSLinux_ZABBIX_Host_Uptime  6.714699e+06  6.713072e+06                    31.0             2.0                1615516140.0         6.714789e+06              2.0                 1615514400.0         6.712982e+06          6.713885e+06, 1627, 685)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables: thresholds_metric_df, metric_window_df

# Candidate components list
candidate_components = ["IG01","IG02","MG01","MG02","Mysql01",
                        "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                        "Redis01","Redis02","apache01","apache02"]

# 1) Restrict metric_window_df to candidate components
window_candidates_df = metric_window_df[metric_window_df['cmdb_id'].isin(candidate_components)].copy()

# 2) Join with thresholds_metric_df to attach p95 and p05
window_with_thresh2 = window_candidates_df.merge(thresholds_metric_df, on=['cmdb_id','kpi_name'], how='left')

# 3) Mark anomalies
window_with_thresh2['anomaly_high'] = window_with_thresh2['value'] > window_with_thresh2['p95']
window_with_thresh2['anomaly_low']  = window_with_thresh2['value'] < window_with_thresh2['p05']

# 4) Find consecutive sequences for each anomaly type
sequences = []  # accumulate sequence dicts

# Helper to process one anomaly type
def extract_sequences(df, anomaly_col, anomaly_type_label):
    df_flag = df[df[anomaly_col]].copy()
    if df_flag.empty:
        return
    # ensure sorted
    df_flag.sort_values(['cmdb_id','kpi_name','timestamp'], inplace=True)
    # group by series
    for (cmdb, kpi), g in df_flag.groupby(['cmdb_id','kpi_name']):
        ts = g['timestamp'].astype(int)
        # compute gap diff in seconds between consecutive timestamps
        diffs = ts.diff().fillna(9999).astype(int)
        # start a new sequence when diff != 60
        seq_id = (diffs != 60).cumsum()
        g = g.assign(seq_id=seq_id.values)
        # aggregate per seq_id
        for sid, seq in g.groupby('seq_id'):
            start_ts = int(seq['timestamp'].min())
            end_ts = int(seq['timestamp'].max())
            length = int(seq.shape[0])
            max_val = float(seq['value'].max())
            min_val = float(seq['value'].min())
            # thresholds (same within series)
            p95 = float(seq['p95'].iloc[0]) if not pd.isna(seq['p95'].iloc[0]) else np.nan
            p05 = float(seq['p05'].iloc[0]) if not pd.isna(seq['p05'].iloc[0]) else np.nan
            # severity
            if anomaly_type_label == 'high':
                severity = np.nan if p95 == 0 or np.isnan(p95) else (max_val / p95)
            else:
                severity = np.nan if min_val == 0 or np.isnan(min_val) else (p05 / min_val)
            sequences.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'anomaly_type': anomaly_type_label,
                'start_timestamp': start_ts,
                'end_timestamp': end_ts,
                'length_in_points': length,
                'p95': p95,
                'p05': p05,
                'max_value_in_seq': max_val,
                'min_value_in_seq': min_val,
                'severity': severity
            })

# extract for both types
extract_sequences(window_with_thresh2, 'anomaly_high', 'high')
extract_sequences(window_with_thresh2, 'anomaly_low', 'low')

sequences_df = pd.DataFrame(sequences)

# 5) Filter out sequences with length_in_points < 2
if not sequences_df.empty:
    sequences_df = sequences_df[sequences_df['length_in_points'] >= 2].copy()
else:
    sequences_df = sequences_df  # remains empty

# 6) Top 20 sequences sorted by length desc, then severity desc
if not sequences_df.empty:
    sequences_df['severity_sort'] = sequences_df['severity'].fillna(-np.inf)  # NaNs become very small for sorting
    top20_sequences_df = sequences_df.sort_values(by=['length_in_points','severity_sort'], ascending=[False,False]) \
                                     .loc[:, ['cmdb_id','kpi_name','anomaly_type','start_timestamp','end_timestamp',
                                              'length_in_points','p95','p05','max_value_in_seq','min_value_in_seq','severity']] \
                                     .head(20) \
                                     .reset_index(drop=True)
else:
    top20_sequences_df = sequences_df.loc[:, ['cmdb_id','kpi_name','anomaly_type','start_timestamp','end_timestamp',
                                              'length_in_points','p95','p05','max_value_in_seq','min_value_in_seq','severity']]

# 7) Small summary counts
num_sequences = int(sequences_df.shape[0])
distinct_components_with_seq = int(sequences_df['cmdb_id'].nunique()) if num_sequences>0 else 0

# Return compact outputs (variables)
top20_sequences_df, num_sequences, distinct_components_with_seq

```
Out[3]:
```
```python
# Compose a concise plain-English summary using existing kernel variables:
# top20_sequences_df, num_sequences, distinct_components_with_seq

import pandas as pd

if top20_sequences_df is None or top20_sequences_df.shape[0] == 0:
    summary_text = (
        "No anomaly sequences (length >= 2) were found in the candidate components during the window."
        f" Total sequences found: {num_sequences}. Distinct components with sequences: {distinct_components_with_seq}."
    )
else:
    # Build up to 5 example lines from the top sequences
    examples = []
    for _, r in top20_sequences_df.head(5).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | type={r['anomaly_type']} | "
            f"start={int(r['start_timestamp'])} end={int(r['end_timestamp'])} | "
            f"len={int(r['length_in_points'])} | max={float(r['max_value_in_seq']):.6g} | "
            f"min={float(r['min_value_in_seq']):.6g} | severity={float(r['severity']):.6g}"
        )

    summary_text = (
        "Anomaly sequence scan summary (sequences length >= 2):\n\n"
        f"- Total sequences found: {num_sequences}\n"
        f"- Distinct components with ≥1 sequence: {distinct_components_with_seq}\n\n"
        "- Top examples (up to 5) from the top-20 sequences sorted by length then severity:\n  "
        + "\n  ".join(examples)
        + "\n\n"
        "Notes: Each example shows cmdb_id | kpi_name | anomaly_type | start_timestamp | end_timestamp | "
        "length_in_points | max_value_in_seq | min_value_in_seq | severity. Timestamps are epoch seconds."
    )

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name anomaly_type  start_timestamp  end_timestamp  length_in_points           p95           p05  max_value_in_seq  min_value_in_seq  severity
0       IG02                        OSLinux-CPU_CPU_CPUidleutil          low       1615516080     1615516140                 2  7.375016e+01  2.093193e+01      1.537470e+01      1.185599e+01  1.765515
1       IG02                             OSLinux-CPU_CPU_CPUWio          low       1615515180     1615515240                 2  1.620916e+00  3.928645e-02      3.741400e-02      2.494900e-02  1.574670
2       IG02                         OSLinux-CPU_CPU_CPUSysTime         high       1615515900     1615515960                 2  6.862123e-01  4.293150e-01      8.288000e-01      6.981310e-01  1.207789
3    Redis02                        OSLinux-CPU_CPU_CPUUserTime          low       1615516140     1615516200                 2  3.913725e-01  2.418000e-01      2.335000e-01      2.044390e-01  1.182749
4       IG02                            OSLinux-CPU_CPU_CPULoad         high       1615516080     1615516140                 2  4.674500e+00  1.000000e+00      5.290000e+00      5.210000e+00  1.131672
5       IG02                        OSLinux-CPU_CPU_CPUUserTime         high       1615516080     1615516140                 2  7.725361e+01  2.539025e+01      8.679655e+01      8.300520e+01  1.123527
6   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...         high       1615515960     1615516020                 2  3.100000e+01  3.000000e+00      3.400000e+01      3.300000e+01  1.096774
7    Mysql01                             OSLinux-CPU_CPU_CPUWio         high       1615516140     1615516200                 2  1.416635e+01  1.031770e+00      1.536510e+01      1.421980e+01  1.084620
8   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...         high       1615515960     1615516020                 2  2.950000e+01  2.000000e+00      3.100000e+01      3.000000e+01  1.050847
9    Mysql01                        OSLinux-CPU_CPU_CPUidleutil          low       1615516140     1615516200                 2  9.803688e+01  8.504770e+01      8.493180e+01      8.367020e+01  1.016463
10      IG02           JVM-Threads_7778_JVM_ThreadCount_Threads         high       1615515840     1615515900                 2  7.550000e+01  6.700000e+01      7.600000e+01      7.600000e+01  1.006623
11   Redis02  redis-Redis_6379_Redis  (mem_fragmentation_ratio)         high       1615516020     1615516080                 2  6.855000e-01  6.700000e-01      6.900000e-01      6.900000e-01  1.006565
12      IG01                            OSLinux-CPU_CPU_CPULoad          low       1615515600     1615515660                 2  1.206500e+00  1.004500e+00      1.000000e+00      1.000000e+00  1.004500
13  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  6.379476e+08  6.214076e+08      6.402738e+08      6.386512e+08  1.003646
14  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  6.759160e+08  6.591909e+08      6.783048e+08      6.766749e+08  1.003534
15  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  6.402499e+08  6.237299e+08      6.424442e+08      6.409380e+08  1.003427
16   Redis01          redis-Redis_6379_Redis  (used_memory_rss)         high       1615516020     1615516080                 2  1.244570e+07  1.205658e+07      1.246822e+07      1.245594e+07  1.001810
17  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  1.409071e+09  1.392857e+09      1.411505e+09      1.409738e+09  1.001727
18  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.792245e+05  9.640230e+05      9.637660e+05      9.632030e+05  1.000851
19  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.802965e+05  9.654295e+05      9.651780e+05      9.646260e+05  1.000833, 77, 11)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing trace_df from kernel

# Allowed components
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# 1) Compute global P95 and P05 for each (cmdb_id, trace_name) across full series (no time filter),
#    but restrict to allowed components (focus).
trace_allowed_full = trace_df[trace_df['cmdb_id'].isin(allowed_components)].copy()

grp = trace_allowed_full.groupby(['cmdb_id','trace_name'])['value']
thresholds_trace_df = grp.quantile(0.95).reset_index().rename(columns={'value':'p95'})
p05_tmp = grp.quantile(0.05).reset_index().rename(columns={'value':'p05'})
thresholds_trace_df = thresholds_trace_df.merge(p05_tmp, on=['cmdb_id','trace_name'])

# 2) Filter trace_df to incident window March 12, 2021 10:00-10:30 (UTC+8) and restrict components
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2021,3,12,10,0,0))
end_dt   = tz.localize(datetime(2021,3,12,10,30,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

trace_window_df = trace_allowed_full[(trace_allowed_full['timestamp'] >= start_ts) &
                                     (trace_allowed_full['timestamp'] <= end_ts)].copy()

# 3) Join trace_window_df with thresholds_trace_df to attach p95/p05
trace_window_with_thresh = trace_window_df.merge(thresholds_trace_df, on=['cmdb_id','trace_name'], how='left')

# 4) Aggregate per (cmdb_id, trace_name)
def agg_trace_group(g):
    count_points = int(g.shape[0])
    count_over = int((g['value'] > g['p95']).sum())
    count_below = int((g['value'] < g['p05']).sum())
    earliest_over = int(g.loc[g['value'] > g['p95'], 'timestamp'].min()) if count_over>0 else pd.NA
    earliest_below = int(g.loc[g['value'] < g['p05'], 'timestamp'].min()) if count_below>0 else pd.NA
    max_val = float(g['value'].max()) if count_points>0 else pd.NA
    min_val = float(g['value'].min()) if count_points>0 else pd.NA
    mean_val = float(g['value'].mean()) if count_points>0 else pd.NA
    return pd.Series({
        'count_points_in_window': count_points,
        'count_over_p95': count_over,
        'earliest_over_p95_timestamp': earliest_over,
        'count_below_p05': count_below,
        'earliest_below_p05_timestamp': earliest_below,
        'max_value_in_window': max_val,
        'min_value_in_window': min_val,
        'mean_value_in_window': mean_val
    })

if trace_window_with_thresh.shape[0] > 0:
    agg_trace_df = trace_window_with_thresh.groupby(['cmdb_id','trace_name']).apply(agg_trace_group).reset_index()
else:
    agg_trace_df = pd.DataFrame(columns=['cmdb_id','trace_name','count_points_in_window','count_over_p95',
                                         'earliest_over_p95_timestamp','count_below_p05','earliest_below_p05_timestamp',
                                         'max_value_in_window','min_value_in_window','mean_value_in_window'])

# 4b) Join with thresholds to include p95/p05 columns
agg_with_thresh_df = agg_trace_df.merge(thresholds_trace_df, on=['cmdb_id','trace_name'], how='left')

# 5) Prepare compact top-20 sorted by count_over_p95 desc, then count_below_p05 desc
agg_with_thresh_df['sort_second'] = agg_with_thresh_df['count_below_p05']
top20_traces_df = agg_with_thresh_df.sort_values(by=['count_over_p95','sort_second'], ascending=[False,False]) \
                                    .loc[:, ['cmdb_id','trace_name','p95','p05',
                                             'count_points_in_window','count_over_p95','earliest_over_p95_timestamp',
                                             'count_below_p05','earliest_below_p05_timestamp',
                                             'max_value_in_window','min_value_in_window','mean_value_in_window']] \
                                    .head(20) \
                                    .reset_index(drop=True)

# 6) Summary counts
total_series_count_trace = int(thresholds_trace_df.shape[0])
anomaly_series_count_trace = int(((agg_with_thresh_df['count_over_p95']>0) | (agg_with_thresh_df['count_below_p05']>0)).sum())

# Return compact results
top20_traces_df, total_series_count_trace, anomaly_series_count_trace

```
Out[4]:
```
```python
# Compose a concise plain-English summary string based on existing kernel results
# Reuse kernel variables: top20_traces_df, total_series_count_trace, anomaly_series_count_trace

examples = []
for _, r in top20_traces_df.head(3).iterrows():
    examples.append(
        f"{r['cmdb_id']} | {r['trace_name']} | p95={float(r['p95']):.6g} p05={float(r['p05']):.6g} | "
        f"points={int(r['count_points_in_window'])} over_p95={int(r['count_over_p95'])} (earliest={int(r['earliest_over_p95_timestamp']) if not pd.isna(r['earliest_over_p95_timestamp']) else 'NA'}) | "
        f"below_p05={int(r['count_below_p05'])} (earliest={int(r['earliest_below_p05_timestamp']) if not pd.isna(r['earliest_below_p05_timestamp']) else 'NA'}) | "
        f"max={float(r['max_value_in_window']):.6g} min={float(r['min_value_in_window']):.6g} mean={float(r['mean_value_in_window']):.6g}"
    )

summary_text = (
    "Trace anomaly scan summary for March 12, 2021 10:00–10:30 (UTC+8):\n\n"
    f"- Global thresholds (P95/P05) were computed per (cmdb_id, trace_name) across the full trace series.\n"
    f"- Total distinct (cmdb_id, trace_name) series evaluated: {total_series_count_trace}.\n"
    f"- Series that showed at least one anomaly in the window (value>p95 or value<p05): {anomaly_series_count_trace}.\n\n"
    "- Top examples from the top-20 aggregated results (first 3 shown):\n  " + "\n  ".join(examples) + "\n\n"
    "Notes: Many top entries are IG01/IG02 self- and to-Tomcat traces (duration and row_count). "
    "Typical patterns: ~30 points in-window per series, with a small number of points exceeding P95 and a small number below P05. "
    "Full top-20 table is available in variable top20_traces_df."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name          p95          p05  count_points_in_window  count_over_p95 earliest_over_p95_timestamp  count_below_p05 earliest_below_p05_timestamp  max_value_in_window  min_value_in_window  mean_value_in_window
0     IG01         trace.self.duration_mean     3.253358     0.512034                    30.0             2.0                1615515960.0              2.0                 1615514400.0             3.572245             0.468784              1.180025
1     IG01          trace.self.duration_p95    11.369605     2.104895                    30.0             2.0                1615516020.0              2.0                 1615514400.0            11.529950             2.088900              4.548848
2     IG01             trace.self.row_count  2510.100000  1894.600000                    30.0             2.0                1615515060.0              2.0                 1615515780.0          2598.000000          1834.000000           2215.000000
3     IG01  trace.to_Tomcat01.duration_mean     3.215496     0.540828                    30.0             2.0                1615516020.0              2.0                 1615514400.0             4.354255             0.423525              1.203933
4     IG01   trace.to_Tomcat01.duration_p95    11.280645     2.118437                    30.0             2.0                1615515960.0              2.0                 1615514400.0            12.095700             2.045500              4.672713
5     IG01  trace.to_Tomcat02.duration_mean     3.003555     0.516542                    30.0             2.0                1615515960.0              2.0                 1615514400.0             3.808393             0.461719              1.196160
6     IG01   trace.to_Tomcat02.duration_p95    11.058030     2.094563                    30.0             2.0                1615515960.0              2.0                 1615514400.0            13.036300             2.083400              4.545970
7     IG01      trace.to_Tomcat02.row_count   306.650000   225.400000                    30.0             2.0                1615515000.0              2.0                 1615515780.0           331.000000           217.000000            270.800000
8     IG01  trace.to_Tomcat03.duration_mean     3.437732     0.449205                    30.0             2.0                1615515960.0              2.0                 1615515540.0             3.649279             0.417957              1.175673
9     IG01   trace.to_Tomcat03.duration_p95    12.672857     2.075338                    30.0             2.0                1615515960.0              2.0                 1615514460.0            16.406750             2.061450              4.745080
10    IG01      trace.to_Tomcat03.row_count   317.100000   236.700000                    30.0             2.0                1615514760.0              2.0                 1615515780.0           322.000000           228.000000            276.033333
11    IG01  trace.to_Tomcat04.duration_mean     2.845704     0.502602                    30.0             2.0                1615516020.0              2.0                 1615514400.0             3.582309             0.435415              1.133342
12    IG01   trace.to_Tomcat04.duration_p95    10.701580     2.087460                    30.0             2.0                1615516020.0              2.0                 1615515060.0            11.454500             2.076600              4.448140
13    IG01      trace.to_Tomcat04.row_count   333.650000   236.350000                    30.0             2.0                1615514940.0              2.0                 1615516080.0           343.000000           233.000000            283.800000
14    IG02         trace.self.duration_mean     3.450861     0.545982                    30.0             2.0                1615516020.0              2.0                 1615515360.0             3.496838             0.469380              1.181193
15    IG02          trace.self.duration_p95    11.052912     2.108828                    30.0             2.0                1615516020.0              2.0                 1615515360.0            12.176550             2.084000              4.564100
16    IG02             trace.self.row_count  2578.200000  1842.900000                    30.0             2.0                1615515240.0              2.0                 1615515960.0          2678.000000          1766.000000           2205.233333
17    IG02  trace.to_Tomcat01.duration_mean     3.130866     0.571107                    30.0             2.0                1615516020.0              2.0                 1615514400.0             3.483015             0.527463              1.162976
18    IG02   trace.to_Tomcat01.duration_p95    10.637015     2.135442                    30.0             2.0                1615515960.0              2.0                 1615514400.0            13.695950             2.107400              4.612510
19    IG02      trace.to_Tomcat01.row_count   325.850000   224.500000                    30.0             2.0                1615515180.0              2.0                 1615515960.0           342.000000           202.000000            274.633333, 189, 179)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: trace_window_df, thresholds_trace_df
tz = pytz.timezone('Asia/Shanghai')

# 1) Filter trace_window_df to IG gateways and to_Tomcat* traces
ig_list = ['IG01','IG02']
tomcat_patterns = ['to_Tomcat01','to_Tomcat02','to_Tomcat03','to_Tomcat04']
pattern_regex = '|'.join(tomcat_patterns)

ig_tomcat_df = trace_window_df[
    (trace_window_df['cmdb_id'].isin(ig_list)) &
    (trace_window_df['trace_name'].str.contains(pattern_regex, na=False))
].copy()

# 2) Join with thresholds_trace_df to attach p95 and p05
ig_tomcat_with_thresh = ig_tomcat_df.merge(thresholds_trace_df, on=['cmdb_id','trace_name'], how='left')

# 3-4) Aggregate per (cmdb_id, trace_name)
def agg_trace_group(g):
    count_points = int(g.shape[0])
    # comparisons using attached p95/p05 per-row
    over_mask = g['value'] > g['p95']
    below_mask = g['value'] < g['p05']
    count_over = int(over_mask.sum())
    count_below = int(below_mask.sum())
    earliest_over = int(g.loc[over_mask, 'timestamp'].min()) if count_over>0 else pd.NA
    earliest_below = int(g.loc[below_mask, 'timestamp'].min()) if count_below>0 else pd.NA
    max_val = float(g['value'].max()) if count_points>0 else pd.NA
    min_val = float(g['value'].min()) if count_points>0 else pd.NA
    mean_val = float(g['value'].mean()) if count_points>0 else pd.NA
    # p95/p05 (should be constant per group)
    p95 = float(g['p95'].iloc[0]) if not pd.isna(g['p95'].iloc[0]) else np.nan
    p05 = float(g['p05'].iloc[0]) if not pd.isna(g['p05'].iloc[0]) else np.nan
    return pd.Series({
        'count_points_in_window': count_points,
        'count_over_p95': count_over,
        'earliest_over_p95_timestamp': earliest_over,
        'count_below_p05': count_below,
        'earliest_below_p05_timestamp': earliest_below,
        'max_value_in_window': max_val,
        'min_value_in_window': min_val,
        'mean_value_in_window': mean_val,
        'p95': p95,
        'p05': p05
    })

if ig_tomcat_with_thresh.shape[0] > 0:
    agg_ig_tomcat = ig_tomcat_with_thresh.groupby(['cmdb_id','trace_name']).apply(agg_trace_group).reset_index()
else:
    agg_ig_tomcat = pd.DataFrame(columns=['cmdb_id','trace_name','count_points_in_window','count_over_p95',
                                         'earliest_over_p95_timestamp','count_below_p05','earliest_below_p05_timestamp',
                                         'max_value_in_window','min_value_in_window','mean_value_in_window','p95','p05'])

# Ensure earliest timestamps are integer epoch seconds or NA; already set as int or pd.NA above

# 5) Sort by earliest_over_p95_timestamp ascending, NaNs last, limit to 50 rows
agg_ig_tomcat_sorted = agg_ig_tomcat.sort_values(by='earliest_over_p95_timestamp', ascending=True, na_position='last') \
                                    .loc[:, ['cmdb_id','trace_name','p95','p05',
                                             'count_points_in_window','count_over_p95','earliest_over_p95_timestamp',
                                             'count_below_p05','earliest_below_p05_timestamp',
                                             'max_value_in_window','min_value_in_window','mean_value_in_window']] \
                                    .head(50) \
                                    .reset_index(drop=True)

# 6) Small summary counts
num_series_found = int(agg_ig_tomcat.shape[0])
num_series_with_anomaly = int(((agg_ig_tomcat['count_over_p95']>0) | (agg_ig_tomcat['count_below_p05']>0)).sum())

# Return compact results (DataFrame and two counts)
agg_ig_tomcat_sorted, num_series_found, num_series_with_anomaly

```
Out[5]:
```
```python
# Compose a concise plain-English summary based on existing kernel variables:
# agg_ig_tomcat_sorted, num_series_found, num_series_with_anomaly

examples = []
for _, r in agg_ig_tomcat_sorted.head(5).iterrows():
    cmdb = r['cmdb_id']
    trace = r['trace_name']
    p95 = float(r['p95']) if not pd.isna(r['p95']) else None
    p05 = float(r['p05']) if not pd.isna(r['p05']) else None
    pts = int(r['count_points_in_window']) if not pd.isna(r['count_points_in_window']) else 0
    over = int(r['count_over_p95']) if not pd.isna(r['count_over_p95']) else 0
    eover = int(r['earliest_over_p95_timestamp']) if not pd.isna(r['earliest_over_p95_timestamp']) else None
    below = int(r['count_below_p05']) if not pd.isna(r['count_below_p05']) else 0
    ebelow = int(r['earliest_below_p05_timestamp']) if not pd.isna(r['earliest_below_p05_timestamp']) else None
    maxv = float(r['max_value_in_window']) if not pd.isna(r['max_value_in_window']) else None
    minv = float(r['min_value_in_window']) if not pd.isna(r['min_value_in_window']) else None
    meanv = float(r['mean_value_in_window']) if not pd.isna(r['mean_value_in_window']) else None
    examples.append(
        f"{cmdb} | {trace} | p95={p95:.6g} p05={p05:.6g} | pts={pts} | over={over} (earliest={eover}) | "
        f"below={below} (earliest={ebelow}) | max={maxv:.6g} min={minv:.6g} mean={meanv:.6g}"
    )

summary = (
    f"IG -> Tomcat trace summary for March 12, 2021 10:00–10:30 (UTC+8):\n"
    f"- Trace series found: {int(num_series_found)} (filtered to IG01/IG02 -> to_Tomcat*).\n"
    f"- Series with ≥1 anomaly in window (value>p95 or value<p05): {int(num_series_with_anomaly)}.\n"
    f"- Example top entries (up to 5):\n  " + "\n  ".join(examples) + "\n\n"
    "Timestamps are epoch seconds. Full compact table is in variable agg_ig_tomcat_sorted."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                       trace_name         p95         p05  count_points_in_window  count_over_p95  earliest_over_p95_timestamp  count_below_p05  earliest_below_p05_timestamp  max_value_in_window  min_value_in_window  mean_value_in_window
0     IG01      trace.to_Tomcat01.row_count  340.250000  235.000000                    30.0             2.0                 1.615515e+09              1.0                  1.615516e+09           350.000000           225.000000            276.800000
1     IG01      trace.to_Tomcat03.row_count  317.100000  236.700000                    30.0             2.0                 1.615515e+09              2.0                  1.615516e+09           322.000000           228.000000            276.033333
2     IG02      trace.to_Tomcat02.row_count  329.450000  226.050000                    30.0             2.0                 1.615515e+09              2.0                  1.615516e+09           339.000000           197.000000            277.066667
3     IG01      trace.to_Tomcat04.row_count  333.650000  236.350000                    30.0             2.0                 1.615515e+09              2.0                  1.615516e+09           343.000000           233.000000            283.800000
4     IG01      trace.to_Tomcat02.row_count  306.650000  225.400000                    30.0             2.0                 1.615515e+09              2.0                  1.615516e+09           331.000000           217.000000            270.800000
..     ...                              ...         ...         ...                     ...             ...                          ...              ...                           ...                  ...                  ...                   ...
19    IG01   trace.to_Tomcat04.duration_p95   10.701580    2.087460                    30.0             2.0                 1.615516e+09              2.0                  1.615515e+09            11.454500             2.076600              4.448140
20    IG01  trace.to_Tomcat01.duration_mean    3.215496    0.540828                    30.0             2.0                 1.615516e+09              2.0                  1.615514e+09             4.354255             0.423525              1.203933
21    IG02  trace.to_Tomcat01.duration_mean    3.130866    0.571107                    30.0             2.0                 1.615516e+09              2.0                  1.615514e+09             3.483015             0.527463              1.162976
22    IG01  trace.to_Tomcat04.duration_mean    2.845704    0.502602                    30.0             2.0                 1.615516e+09              2.0                  1.615514e+09             3.582309             0.435415              1.133342
23    IG02   trace.to_Tomcat02.duration_p95   11.170470    2.095328                    30.0             2.0                 1.615516e+09              2.0                  1.615515e+09            11.225250             2.057050              4.571842

[24 rows x 12 columns], 24, 24)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse metric_df from kernel state

# Candidate components as requested
candidate_components = ["IG01","IG02","MG01","MG02","Mysql01","Mysql02",
                        "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                        "Redis01","Redis02","apache01","apache02"]

# Time window (UTC+8) March 12, 2021 10:00 to 10:30
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2021,3,12,10,0,0))
end_dt   = tz.localize(datetime(2021,3,12,10,30,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# 1) Compute global p95 and p05 per (cmdb_id, kpi_name) across full series for candidate components
metric_candidates_full = metric_df[metric_df['cmdb_id'].isin(candidate_components)].copy()

grp_full = metric_candidates_full.groupby(['cmdb_id','kpi_name'])['value']
thresholds_metric_df = grp_full.quantile(0.95).reset_index().rename(columns={'value':'p95'})
p05_tmp = grp_full.quantile(0.05).reset_index().rename(columns={'value':'p05'})
thresholds_metric_df = thresholds_metric_df.merge(p05_tmp, on=['cmdb_id','kpi_name'])

# 2) Filter metric_df to the incident window and candidate components
metric_window_candidates_df = metric_candidates_full[
    (metric_candidates_full['timestamp'] >= start_ts) &
    (metric_candidates_full['timestamp'] <= end_ts)
].copy()

# 3) Join windowed rows with thresholds to attach p95/p05
window_with_thresh = metric_window_candidates_df.merge(thresholds_metric_df, on=['cmdb_id','kpi_name'], how='left')

# 4) Mark anomalies
window_with_thresh['anomaly_high'] = window_with_thresh['value'] > window_with_thresh['p95']
window_with_thresh['anomaly_low']  = window_with_thresh['value'] < window_with_thresh['p05']

# 5) Extract consecutive sequences for each anomaly type
sequences = []

def extract_sequences(df, anomaly_col, anomaly_label):
    df_flag = df[df[anomaly_col]].copy()
    if df_flag.empty:
        return
    df_flag.sort_values(['cmdb_id','kpi_name','timestamp'], inplace=True)
    # Ensure integer timestamps
    df_flag['timestamp'] = df_flag['timestamp'].astype(int)
    for (cmdb, kpi), g in df_flag.groupby(['cmdb_id','kpi_name']):
        ts = g['timestamp']
        diffs = ts.diff().fillna(9999).astype(int)
        seq_id = (diffs != 60).cumsum()
        g = g.assign(seq_id=seq_id.values)
        for sid, seq in g.groupby('seq_id'):
            start_ts_seq = int(seq['timestamp'].min())
            end_ts_seq = int(seq['timestamp'].max())
            length = int(seq.shape[0])
            max_val = float(seq['value'].max())
            min_val = float(seq['value'].min())
            # thresholds (should be same within group)
            p95 = float(seq['p95'].iloc[0]) if not pd.isna(seq['p95'].iloc[0]) else np.nan
            p05 = float(seq['p05'].iloc[0]) if not pd.isna(seq['p05'].iloc[0]) else np.nan
            # severity calculation
            if anomaly_label == 'high':
                severity = np.nan if (p95 == 0 or np.isnan(p95)) else (max_val / p95)
            else:
                severity = np.nan if (min_val == 0 or np.isnan(min_val)) else (p05 / min_val)
            sequences.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'anomaly_type': anomaly_label,
                'start_timestamp': start_ts_seq,
                'end_timestamp': end_ts_seq,
                'length_in_points': length,
                'p95': p95,
                'p05': p05,
                'max_value_in_seq': max_val,
                'min_value_in_seq': min_val,
                'severity': severity
            })

# Extract for both high and low anomalies
extract_sequences(window_with_thresh, 'anomaly_high', 'high')
extract_sequences(window_with_thresh, 'anomaly_low', 'low')

sequences_df = pd.DataFrame(sequences)

# 6) Filter sequences with length >= 2
if not sequences_df.empty:
    sequences_df = sequences_df[sequences_df['length_in_points'] >= 2].copy()
else:
    sequences_df = sequences_df  # remain empty

# Ensure integer timestamps
if not sequences_df.empty:
    sequences_df['start_timestamp'] = sequences_df['start_timestamp'].astype(int)
    sequences_df['end_timestamp'] = sequences_df['end_timestamp'].astype(int)

# 7) Frequency table of sequence start timestamps (top 10 by count descending)
if not sequences_df.empty:
    start_freq = sequences_df.groupby('start_timestamp').size().reset_index(name='sequence_count')
    start_freq_sorted = start_freq.sort_values(by='sequence_count', ascending=False).head(10).reset_index(drop=True)
else:
    start_freq_sorted = pd.DataFrame(columns=['start_timestamp','sequence_count'])

# 8) For top 2 start_timestamps, list sequences that start at those timestamps
top_start_ts = start_freq_sorted['start_timestamp'].head(2).tolist()
if len(top_start_ts) == 0:
    top_sequences_for_top2_df = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_type','start_timestamp','end_timestamp',
                                                     'length_in_points','p95','p05','max_value_in_seq','min_value_in_seq','severity'])
else:
    seqs_top2 = sequences_df[sequences_df['start_timestamp'].isin(top_start_ts)].copy()
    # sort for readability: by start_timestamp, length desc, severity desc
    seqs_top2 = seqs_top2.sort_values(by=['start_timestamp','length_in_points','severity'], ascending=[True,False,False])
    # limit rows per timestamp to 50 and overall to 100
    seqs_top2_limited = seqs_top2.groupby('start_timestamp').head(50).reset_index(drop=True)
    top_sequences_for_top2_df = seqs_top2_limited.head(100).loc[:, ['cmdb_id','kpi_name','anomaly_type','start_timestamp','end_timestamp',
                                                                    'length_in_points','p95','p05','max_value_in_seq','min_value_in_seq','severity']].reset_index(drop=True)

# 9) Summary counts
total_sequences_count = int(sequences_df.shape[0]) if not sequences_df.empty else 0
distinct_components_with_sequences = int(sequences_df['cmdb_id'].nunique()) if total_sequences_count>0 else 0

# Return compact outputs as variables
start_freq_sorted, top_sequences_for_top2_df, total_sequences_count, distinct_components_with_sequences

```
Out[6]:
```
# Compose a concise plain-English summary of the sequence analysis results
# Reuse kernel variables: start_freq_sorted, top_sequences_for_top2_df, total_sequences_count, distinct_components_with_sequences

# Prepare top-10 start timestamp frequencies
top10_starts = []
if 'start_timestamp' in start_freq_sorted.columns:
    for _, r in start_freq_sorted.head(10).iterrows():
        top10_starts.append((int(r['start_timestamp']), int(r['sequence_count'])))

# Prepare examples for the top 2 start timestamps
examples_lines = []
if not top_sequences_for_top2_df.empty:
    for _, r in top_sequences_for_top2_df.iterrows():
        examples_lines.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | {r['anomaly_type']} | "
            f"start={int(r['start_timestamp'])} end={int(r['end_timestamp'])} len={int(r['length_in_points'])} "
            f"p95={float(r['p95']):.6g} p05={float(r['p05']):.6g} max={float(r['max_value_in_seq']):.6g} min={float(r['min_value_in_seq']):.6g} sev={float(r['severity']):.6g}"
        )

# Build summary string
summary_lines = []
summary_lines.append(f"Anomaly sequence analysis (sequences length >= 2) during March 12, 2021 10:00–10:30 (UTC+8):")
summary_lines.append(f"- Total sequences found: {int(total_sequences_count)}")
summary_lines.append(f"- Distinct components with sequences: {int(distinct_components_with_sequences)}")
summary_lines.append("")
summary_lines.append("- Top start timestamps by number of sequences (top 10):")
for ts, cnt in top10_starts:
    summary_lines.append(f"  - {ts} : {cnt} sequences")
summary_lines.append("")
summary_lines.append("- Examples of sequences starting at the top 2 timestamps:")
# Limit examples shown to first 20 for compactness
for line in examples_lines[:20]:
    summary_lines.append(f"  - {line}")
summary_text = "\n".join(summary_lines)

# Display the summary string (compact)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(   start_timestamp  sequence_count
0       1615514400              32
1       1615516140              31
2       1615516080               7
3       1615516020               5
4       1615515600               3
5       1615515960               2
6       1615514640               1
7       1615515180               1
8       1615515900               1
9       1615515720               1,      cmdb_id                                           kpi_name anomaly_type  start_timestamp  end_timestamp  length_in_points           p95           p05  max_value_in_seq  min_value_in_seq  severity
0   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.792245e+05  9.640230e+05          963766.0          963203.0  1.000851
1   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.802965e+05  9.654295e+05          965178.0          964626.0  1.000833
2   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...          low       1615514400     1615514460                 2  6.402499e+08  6.237299e+08       623531253.0       623213980.0  1.000828
3   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.786430e+05  9.637095e+05          963480.0          962934.0  1.000805
4   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...          low       1615514400     1615514460                 2  6.759160e+08  6.591909e+08       659014219.0       658722164.0  1.000712
..       ...                                                ...          ...              ...            ...               ...           ...           ...               ...               ...       ...
58  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime         high       1615516140     1615516200                 2  6.714789e+06  6.713163e+06         6714879.0         6714819.0  1.000013
59      IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime         high       1615516140     1615516200                 2  6.715161e+06  6.713538e+06         6715251.0         6715191.0  1.000013
60   Mysql02                 OSLinux-OSLinux_ZABBIX_Host_Uptime         high       1615516140     1615516200                 2  6.714697e+06  6.713128e+06         6714786.0         6714724.0  1.000013
61      IG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime         high       1615516140     1615516200                 2  6.715160e+06  6.713577e+06         6715248.0         6715187.0  1.000013
62   Redis02  redis-Redis_6379_Redis  (total_connections_rec...         high       1615516140     1615516200                 2  7.389944e+07  7.389225e+07        73899858.0        73899573.0  1.000006

[63 rows x 11 columns], 85, 12)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse existing log_df from kernel

# Candidate components
candidate_components = ["Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                        "IG01","IG02","MG01","MG02","Mysql01","Mysql02",
                        "Redis01","Redis02","apache01","apache02"]

# Time window (UTC+8) March 12, 2021 10:00 to 10:30
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2021,3,12,10,0,0))
end_dt   = tz.localize(datetime(2021,3,12,10,30,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# 1) Restrict log_df to candidate components (for threshold computation we consider only these components)
log_candidates_full = log_df[log_df['cmdb_id'].isin(candidate_components)].copy()

# 2) Compute global P95 and P05 for each (cmdb_id, log_name) across the full available series
grp_full = log_candidates_full.groupby(['cmdb_id','log_name'])['value']
thresholds_log_df = grp_full.quantile(0.95).reset_index().rename(columns={'value':'p95'})
p05_tmp = grp_full.quantile(0.05).reset_index().rename(columns={'value':'p05'})
thresholds_log_df = thresholds_log_df.merge(p05_tmp, on=['cmdb_id','log_name'])

# 3) Filter log_df to the incident window and join with thresholds_log_df
log_window_df = log_candidates_full[(log_candidates_full['timestamp'] >= start_ts) &
                                   (log_candidates_full['timestamp'] <= end_ts)].copy()

log_window_with_thresh = log_window_df.merge(thresholds_log_df, on=['cmdb_id','log_name'], how='left')

# 4) Aggregate per (cmdb_id, log_name)
def agg_log_group(g):
    count_points = int(g.shape[0])
    # comparisons using attached p95/p05 per-row
    over_mask = g['value'] > g['p95']
    below_mask = g['value'] < g['p05']
    count_over = int(over_mask.sum())
    count_below = int(below_mask.sum())
    earliest_over = int(g.loc[over_mask, 'timestamp'].min()) if count_over>0 else pd.NA
    earliest_below = int(g.loc[below_mask, 'timestamp'].min()) if count_below>0 else pd.NA
    max_val = float(g['value'].max()) if count_points>0 else pd.NA
    min_val = float(g['value'].min()) if count_points>0 else pd.NA
    mean_val = float(g['value'].mean()) if count_points>0 else pd.NA
    return pd.Series({
        'count_points_in_window': count_points,
        'count_over_p95': count_over,
        'earliest_over_p95_timestamp': earliest_over,
        'count_below_p05': count_below,
        'earliest_below_p05_timestamp': earliest_below,
        'max_value_in_window': max_val,
        'min_value_in_window': min_val,
        'mean_value_in_window': mean_val
    })

if log_window_with_thresh.shape[0] > 0:
    agg_log_df = log_window_with_thresh.groupby(['cmdb_id','log_name']).apply(agg_log_group).reset_index()
else:
    agg_log_df = pd.DataFrame(columns=['cmdb_id','log_name','count_points_in_window','count_over_p95',
                                       'earliest_over_p95_timestamp','count_below_p05','earliest_below_p05_timestamp',
                                       'max_value_in_window','min_value_in_window','mean_value_in_window'])

# 4b) Join with thresholds to include p95/p05 columns
agg_with_thresh_log_df = agg_log_df.merge(thresholds_log_df, on=['cmdb_id','log_name'], how='left')

# 5) Prepare compact top-20 sorted by count_over_p95 desc, then count_below_p05 desc
agg_with_thresh_log_df['sort_second'] = agg_with_thresh_log_df['count_below_p05']
top20_logs_df = agg_with_thresh_log_df.sort_values(by=['count_over_p95','sort_second'], ascending=[False,False]) \
                                      .loc[:, ['cmdb_id','log_name','p95','p05',
                                               'count_points_in_window','count_over_p95','earliest_over_p95_timestamp',
                                               'count_below_p05','earliest_below_p05_timestamp',
                                               'max_value_in_window','min_value_in_window','mean_value_in_window']] \
                                      .head(20) \
                                      .reset_index(drop=True)

# 6) Summary counts
total_series_count_log = int(thresholds_log_df.shape[0])
anomaly_series_count_log = int(((agg_with_thresh_log_df['count_over_p95']>0) | (agg_with_thresh_log_df['count_below_p05']>0)).sum())

# Return compact results
top20_logs_df, total_series_count_log, anomaly_series_count_log

```
Out[7]:
```
# Compose a plain-English summary string based on existing kernel results (variables: top20_logs_df, total_series_count_log, anomaly_series_count_log)
import pandas as pd
import math

examples = []
if 'top20_logs_df' in globals() and isinstance(top20_logs_df, pd.DataFrame) and not top20_logs_df.empty:
    for _, r in top20_logs_df.head(5).iterrows():
        cmdb = r['cmdb_id']
        logn = r['log_name']
        p95 = (f"{float(r['p95']):.6g}" if not pd.isna(r['p95']) else "NA")
        p05 = (f"{float(r['p05']):.6g}" if not pd.isna(r['p05']) else "NA")
        pts = int(r['count_points_in_window']) if not pd.isna(r['count_points_in_window']) else 0
        over = int(r['count_over_p95']) if not pd.isna(r['count_over_p95']) else 0
        eover = int(r['earliest_over_p95_timestamp']) if not pd.isna(r['earliest_over_p95_timestamp']) else "NA"
        below = int(r['count_below_p05']) if not pd.isna(r['count_below_p05']) else 0
        ebelow = int(r['earliest_below_p05_timestamp']) if not pd.isna(r['earliest_below_p05_timestamp']) else "NA"
        maxv = (f"{float(r['max_value_in_window']):.6g}" if not pd.isna(r['max_value_in_window']) else "NA")
        minv = (f"{float(r['min_value_in_window']):.6g}" if not pd.isna(r['min_value_in_window']) else "NA")
        meanv = (f"{float(r['mean_value_in_window']):.6g}" if not pd.isna(r['mean_value_in_window']) else "NA")
        examples.append(
            f"{cmdb} | {logn} | p95={p95} p05={p05} | pts={pts} | over={over} (earliest={eover}) | "
            f"below={below} (earliest={ebelow}) | max={maxv} min={minv} mean={meanv}"
        )

total_series = int(total_series_count_log) if 'total_series_count_log' in globals() else "NA"
anomalous_series = int(anomaly_series_count_log) if 'anomaly_series_count_log' in globals() else "NA"

summary_text = (
    "Log anomaly summary for March 12, 2021 10:00–10:30 (UTC+8):\n\n"
    f"- Total distinct (cmdb_id, log_name) series evaluated: {total_series}\n"
    f"- Series with at least one anomaly in the window (value>p95 or value<p05): {anomalous_series}\n\n"
    "- Top findings (up to 5 examples from the top-20 sorted by anomalies):\n  "
    + ("\n  ".join(examples) if examples else "No results available.")
    + "\n\nNotes: Timestamps are epoch seconds. Full compact table is available in variable top20_logs_df."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name       p95      p05  count_points_in_window  count_over_p95 earliest_over_p95_timestamp  count_below_p05 earliest_below_p05_timestamp  max_value_in_window  min_value_in_window  mean_value_in_window
0   Tomcat01     log.row_count   661.500  452.000                    31.0             2.0                1615514760.0              2.0                 1615516020.0               697.00                 9.00            534.096774
1   Tomcat02     log.row_count   626.500  458.000                    31.0             2.0                1615515240.0              2.0                 1615515960.0               653.00                10.00            531.354839
2   Tomcat03     log.row_count   634.500  466.500                    31.0             2.0                1615514760.0              2.0                 1615514520.0               669.00                 7.00            532.741935
3   Tomcat04  log.gc_pause_max     0.107    0.023                    27.0             2.0                1615514520.0              2.0                 1615514760.0                 0.12                 0.02              0.064074
4   Tomcat04     log.row_count   638.000  468.500                    31.0             2.0                1615514700.0              2.0                 1615516140.0               641.00                 9.00            543.161290
5   apache01     log.row_count  1241.500  929.500                    31.0             2.0                1615514760.0              2.0                 1615515960.0              1324.00                13.00           1068.548387
6   apache02     log.row_count  1242.000  935.500                    31.0             2.0                1615514760.0              2.0                 1615516020.0              1314.00                14.00           1071.290323
7   Tomcat01  log.gc_pause_max     0.078    0.020                    25.0             2.0                  1615514760              0.0                         <NA>                 0.09                 0.02              0.056000
8   Tomcat02  log.gc_pause_max     0.120    0.038                    29.0             1.0                1615515360.0              2.0                 1615515120.0                 0.14                 0.03              0.076897
9   Tomcat03  log.gc_pause_max     0.080    0.020                    25.0             1.0                  1615514820              0.0                         <NA>                 0.09                 0.02              0.053600
10   Mysql02   log.error_count     0.000    0.000                     1.0             0.0                        <NA>              0.0                         <NA>                 0.00                 0.00              0.000000
11   Mysql02     log.row_count     3.000    3.000                     1.0             0.0                        <NA>              0.0                         <NA>                 3.00                 3.00              3.000000
12  Tomcat01   log.error_count     0.000    0.000                    31.0             0.0                        <NA>              0.0                         <NA>                 0.00                 0.00              0.000000
13  Tomcat02   log.error_count     0.000    0.000                    31.0             0.0                        <NA>              0.0                         <NA>                 0.00                 0.00              0.000000
14  Tomcat03   log.error_count     0.000    0.000                    31.0             0.0                        <NA>              0.0                         <NA>                 0.00                 0.00              0.000000
15  Tomcat04   log.error_count     0.000    0.000                    31.0             0.0                        <NA>              0.0                         <NA>                 0.00                 0.00              0.000000
16  apache01   log.error_count     0.000    0.000                    31.0             0.0                        <NA>              0.0                         <NA>                 0.00                 0.00              0.000000
17  apache02   log.error_count     0.000    0.000                    31.0             0.0                        <NA>              0.0                         <NA>                 0.00                 0.00              0.000000, 18, 10)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse sequences_df from kernel (created earlier). If not present, create empty DataFrame to avoid errors.
try:
    sequences_df
except NameError:
    sequences_df = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_type','start_timestamp','end_timestamp',
                                         'length_in_points','p95','p05','max_value_in_seq','min_value_in_seq','severity'])

# Target start timestamps
target_starts = [1615514400, 1615516140]

# 1) Filter sequences to the two start timestamps
seq_filtered = sequences_df[sequences_df['start_timestamp'].isin(target_starts)].copy()

# Ensure numeric types
if not seq_filtered.empty:
    seq_filtered['length_in_points'] = pd.to_numeric(seq_filtered['length_in_points'], errors='coerce')
    seq_filtered['severity'] = pd.to_numeric(seq_filtered['severity'], errors='coerce')
    seq_filtered['start_timestamp'] = seq_filtered['start_timestamp'].astype(int)

# 2) Group by start_timestamp and cmdb_id to compute aggregates
def top_kpis_string(subdf, topn=3):
    counts = subdf['kpi_name'].value_counts().head(topn)
    return "; ".join([f"{k}({int(v)})" for k,v in counts.items()])

if seq_filtered.empty:
    agg_compact_df = pd.DataFrame(columns=['start_timestamp','cmdb_id','sequence_count','max_severity',
                                          'avg_severity','median_length','top_kpis'])
    summary_stats_df = pd.DataFrame(columns=['start_timestamp','distinct_components','top_component','top_component_count'])
else:
    grp = seq_filtered.groupby(['start_timestamp','cmdb_id'])
    agg = grp.agg(
        sequence_count = ('kpi_name','size'),
        max_severity    = ('severity', lambda s: float(np.nanmax(s.values)) if s.notna().any() else np.nan),
        avg_severity    = ('severity', lambda s: float(np.nanmean(s.values)) if s.notna().any() else np.nan),
        median_length   = ('length_in_points', lambda x: float(np.nanmedian(x.values)) if x.notna().any() else np.nan)
    ).reset_index()

    # compute top_kpis per group
    top_kpis = grp.apply(lambda g: top_kpis_string(g, topn=3)).reset_index().rename(columns={0:'top_kpis'})

    agg_compact_df = agg.merge(top_kpis, on=['start_timestamp','cmdb_id'], how='left')

    # Sort by start_timestamp asc then sequence_count desc and limit to 50 rows
    agg_compact_df = agg_compact_df.sort_values(by=['start_timestamp','sequence_count'], ascending=[True,False]) \
                                   .loc[:, ['start_timestamp','cmdb_id','sequence_count','max_severity','avg_severity','median_length','top_kpis']] \
                                   .head(50) \
                                   .reset_index(drop=True)

    # 3) Summary: for each target start timestamp, number of distinct components and top component by sequence_count
    summary_rows = []
    for ts in target_starts:
        df_ts = agg_compact_df[agg_compact_df['start_timestamp']==ts]
        distinct_components = int(df_ts['cmdb_id'].nunique())
        if df_ts.shape[0] > 0:
            top_row = df_ts.sort_values(by='sequence_count', ascending=False).iloc[0]
            top_component = top_row['cmdb_id']
            top_count = int(top_row['sequence_count'])
        else:
            top_component = pd.NA
            top_count = 0
        summary_rows.append({
            'start_timestamp': int(ts),
            'distinct_components': distinct_components,
            'top_component': top_component,
            'top_component_count': top_count
        })
    summary_stats_df = pd.DataFrame(summary_rows)

# Return compact aggregated table and the summary table
agg_compact_df, summary_stats_df

```
Out[8]:
```
summary = (
    "Anomaly sequence summary for the two start timestamps:\n\n"
    "Timestamp 1615514400:\n"
    "- Total sequences starting at this timestamp: 32\n"
    "- Distinct components with sequences: 12\n"
    "- Component with highest sequence_count: Redis01 (9 sequences)\n\n"
    "Timestamp 1615516140:\n"
    "- Total sequences starting at this timestamp: 31\n"
    "- Distinct components with sequences: 12\n    (top component: Redis02 with 8 sequences)\n\n"
    "Notes: Most components contribute a small number of sequences (1–2), while Redis nodes dominate the activity at these two timestamps."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    start_timestamp   cmdb_id  sequence_count  max_severity  avg_severity  median_length                                           top_kpis
0        1615514400   Redis01               9      1.000083      1.000046            2.0  OSLinux-OSLinux_ZABBIX_Host_Uptime(1); redis-R...
1        1615514400   Redis02               9      1.000076      1.000043            2.0  OSLinux-OSLinux_ZABBIX_Host_Uptime(1); redis-R...
2        1615514400  Tomcat01               2      1.000367      1.000349            2.0  Tomcat-Requests_7441-"http-nio-8003"_Processin...
3        1615514400  Tomcat02               2      1.000833      1.000772            2.0  Tomcat-Requests_7441-"http-nio-8003"_Processin...
4        1615514400  Tomcat03               2      1.000805      1.000754            2.0  Tomcat-Requests_7441-"http-nio-8003"_Processin...
..              ...       ...             ...           ...           ...            ...                                                ...
19       1615516140      IG01               1      1.000013      1.000013            2.0              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)
20       1615516140      IG02               1      1.000013      1.000013            2.0              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)
21       1615516140   Mysql02               1      1.000013      1.000013            2.0              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)
22       1615516140  apache01               1      1.000013      1.000013            2.0              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)
23       1615516140  apache02               1      1.000013      1.000013            2.0              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)

[24 rows x 7 columns],    start_timestamp  distinct_components top_component  top_component_count
0       1615514400                   12       Redis01                    9
1       1615516140                   12       Redis02                    8)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables: sequences_df, trace_window_df, thresholds_trace_df
# Ensure they exist
try:
    sequences_df
    trace_window_df
    thresholds_trace_df
except NameError as e:
    raise

# Target start timestamps
target_starts = [1615514400, 1615516140]

# 1) Filter sequences to the two start timestamps
seq_target_df = sequences_df[sequences_df['start_timestamp'].isin(target_starts)].copy()

# Normalize numeric columns
if not seq_target_df.empty:
    seq_target_df['length_in_points'] = pd.to_numeric(seq_target_df['length_in_points'], errors='coerce')
    seq_target_df['severity'] = pd.to_numeric(seq_target_df['severity'], errors='coerce')
    seq_target_df['start_timestamp'] = seq_target_df['start_timestamp'].astype(int)

# 2) Aggregate per (start_timestamp, cmdb_id)
def top_kpis_str(s, topn=3):
    counts = s.value_counts().head(topn)
    return "; ".join([f"{k}({int(v)})" for k,v in counts.items()])

if seq_target_df.empty:
    agg_comp = pd.DataFrame(columns=['start_timestamp','cmdb_id','sequence_count','max_severity','avg_severity','top_kpis'])
else:
    grp = seq_target_df.groupby(['start_timestamp','cmdb_id'])
    agg = grp.agg(
        sequence_count = ('kpi_name','size'),
        max_severity    = ('severity', lambda x: float(np.nanmax(x.values)) if x.notna().any() else np.nan),
        avg_severity    = ('severity', lambda x: float(np.nanmean(x.values)) if x.notna().any() else np.nan)
    ).reset_index()
    topk = grp['kpi_name'].apply(lambda s: top_kpis_str(s, topn=3)).reset_index().rename(columns={'kpi_name':'top_kpis'})
    agg_comp = agg.merge(topk, on=['start_timestamp','cmdb_id'], how='left')

# 3) For each (timestamp, component) find related traces in trace_window_df (trace names containing 'to_{component}')
# and join with thresholds_trace_df to identify anomalies at or before the component start timestamp.
related_cols = ['cmdb_id','trace_name','timestamp','value','p95','p05']

results = []
for _, row in agg_comp.iterrows():
    ts = int(row['start_timestamp'])
    comp = row['cmdb_id']
    # pattern 'to_{comp}'
    pat = f"to_{comp}"
    # Find related trace rows in trace_window_df where trace_name contains the pattern
    traces_rel = trace_window_df[trace_window_df['trace_name'].str.contains(pat, na=False)].copy()
    if traces_rel.empty:
        trace_series_examined = 0
        trace_series_with_anom = 0
        earliest_trace_anom_ts = pd.NA
        examples_trace_names = ""
    else:
        # Attach thresholds for traces (join on cmdb_id, trace_name)
        traces_rel = traces_rel.merge(thresholds_trace_df, on=['cmdb_id','trace_name'], how='left')
        # Compute anomaly points: value>p95 or value<p05
        traces_rel['is_anom'] = ((traces_rel['value'] > traces_rel['p95']) | (traces_rel['value'] < traces_rel['p05']))
        # Only consider anomaly points with timestamp <= component start ts
        anom_before = traces_rel[(traces_rel['is_anom']) & (traces_rel['timestamp'] <= ts)].copy()
        # Count distinct trace series examined (distinct trace_name per cmdb_id)
        trace_series_examined = int(traces_rel[['cmdb_id','trace_name']].drop_duplicates().shape[0])
        # For each trace series, determine if it had at least one anomaly <= ts
        if anom_before.empty:
            trace_series_with_anom = 0
            earliest_trace_anom_ts = pd.NA
            examples_trace_names = ""
        else:
            trace_series_with_anom = int(anom_before[['cmdb_id','trace_name']].drop_duplicates().shape[0])
            earliest_trace_anom_ts = int(anom_before['timestamp'].min())
            examples_trace_names = "; ".join(anom_before['trace_name'].drop_duplicates().head(5).tolist())
    results.append({
        'start_timestamp': ts,
        'cmdb_id': comp,
        'sequence_count': int(row['sequence_count']),
        'max_severity': float(row['max_severity']) if not pd.isna(row['max_severity']) else np.nan,
        'avg_severity': float(row['avg_severity']) if not pd.isna(row['avg_severity']) else np.nan,
        'top_kpis': row.get('top_kpis', ""),
        'trace_series_examined': trace_series_examined,
        'trace_series_with_anomaly_before_or_at_start': trace_series_with_anom,
        'earliest_trace_anomaly_timestamp': earliest_trace_anom_ts,
        'examples_of_trace_names_with_anomalies': examples_trace_names
    })

corr_df = pd.DataFrame(results)

# 5) Sort and limit to 50 rows
if not corr_df.empty:
    corr_df = corr_df.sort_values(by=['start_timestamp','sequence_count'], ascending=[True,False]) \
                     .head(50).reset_index(drop=True)
else:
    corr_df = corr_df

# 6) Summary: for each timestamp, find top component(s) with max sequence_count and report if they had trace anomaly at/before start
summary_rows = []
for ts in target_starts:
    df_ts = corr_df[corr_df['start_timestamp']==ts]
    if df_ts.empty:
        summary_rows.append({
            'start_timestamp': int(ts),
            'top_components': "",
            'had_trace_anomaly': "",
            'earliest_trace_anomaly_timestamp': pd.NA
        })
        continue
    max_cnt = int(df_ts['sequence_count'].max())
    top_comps = df_ts[df_ts['sequence_count']==max_cnt]
    comps_list = top_comps['cmdb_id'].tolist()
    # For these components, determine if any had trace_series_with_anomaly_before_or_at_start >0 and earliest timestamp
    had_anom_flags = []
    earliest_ts_list = []
    for comp in comps_list:
        rec = top_comps[top_comps['cmdb_id']==comp].iloc[0]
        had = 'yes' if rec['trace_series_with_anomaly_before_or_at_start']>0 else 'no'
        had_anom_flags.append(had)
        earliest_ts_list.append(rec['earliest_trace_anomaly_timestamp'] if not pd.isna(rec['earliest_trace_anomaly_timestamp']) else pd.NA)
    # format results
    top_components_str = "; ".join([f"{c}({max_cnt})" for c in comps_list])
    had_anom_str = "; ".join(had_anom_flags)
    earliest_str = "; ".join([str(int(x)) if not pd.isna(x) else "NA" for x in earliest_ts_list])
    summary_rows.append({
        'start_timestamp': int(ts),
        'top_components': top_components_str,
        'had_trace_anomaly': had_anom_str,
        'earliest_trace_anomaly_timestamp': earliest_str
    })

summary_df = pd.DataFrame(summary_rows)

# Return compact correlation table and the summary
corr_df, summary_df

```
Out[9]:
```
```python
# Compose a concise plain-English summary from existing kernel results:
# Uses corr_df and summary_df produced earlier.

import pandas as pd

# Fallback if variables missing
try:
    corr_df
    summary_df
except NameError:
    corr_df = pd.DataFrame()
    summary_df = pd.DataFrame()

lines = []
lines.append("Correlation summary between anomaly sequences and related traces (timestamps: 1615514400, 1615516140):")
lines.append("")

for _, srow in summary_df.sort_values('start_timestamp').iterrows():
    ts = int(srow['start_timestamp'])
    top_comps = srow['top_components']
    had_trace = srow['had_trace_anomaly']
    earliest = srow['earliest_trace_anomaly_timestamp']
    # totals for this timestamp from corr_df
    df_ts = corr_df[corr_df['start_timestamp'] == ts]
    total_sequences = int(df_ts['sequence_count'].sum()) if not df_ts.empty else 0
    distinct_components = int(df_ts['cmdb_id'].nunique()) if not df_ts.empty else 0

    lines.append(f"Timestamp {ts}:")
    lines.append(f"- Total sequences starting at this timestamp: {total_sequences}")
    lines.append(f"- Distinct components with sequences: {distinct_components}")
    lines.append(f"- Top component(s) by sequence count: {top_comps}")
    lines.append(f"- Top component(s) had trace anomaly before-or-at-start? {had_trace} (earliest trace anomaly ts: {earliest})")
    # show up to 3 example components that had related trace anomalies before-or-at-start
    affected = df_ts[df_ts['trace_series_with_anomaly_before_or_at_start']>0]
    if not affected.empty:
        ex = []
        for _, r in affected.head(3).iterrows():
            ex.append(f"{r['cmdb_id']} (trace_series_with_anom={int(r['trace_series_with_anomaly_before_or_at_start'])}, earliest={r['earliest_trace_anomaly_timestamp']})")
        lines.append(f"- Examples of components with related trace anomalies at/ before start: {', '.join(ex)}")
    else:
        lines.append("- No components had related trace anomalies at or before sequence start for this timestamp.")
    lines.append("")

# Brief overall observation
lines.append("Overall observations:")
lines.append("- Redis nodes (Redis01/Redis02) produced the largest number of sequences at these timestamps,")
lines.append("  but they did not have related trace anomalies at or before the sequence starts.")
lines.append("- Several Tomcat components (e.g., Tomcat01, Tomcat02, Tomcat03) show sequences that correlate with")
lines.append("  related traces that had anomalies at the same timestamp (earliest_trace_anomaly_timestamp == sequence start).")
lines.append("")
lines.append("Note: Rows report counts and earliest anomaly timestamps (epoch seconds). Full correlation table is in variable corr_df and summary table is in summary_df.")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(    start_timestamp   cmdb_id  sequence_count  max_severity  avg_severity                                           top_kpis  trace_series_examined  trace_series_with_anomaly_before_or_at_start earliest_trace_anomaly_timestamp             examples_of_trace_names_with_anomalies
0        1615514400   Redis01               9      1.000083      1.000046  OSLinux-OSLinux_ZABBIX_Host_Uptime(1); redis-R...                      0                                             0                             <NA>                                                   
1        1615514400   Redis02               9      1.000076      1.000043  OSLinux-OSLinux_ZABBIX_Host_Uptime(1); redis-R...                      0                                             0                             <NA>                                                   
2        1615514400  Tomcat01               2      1.000367      1.000349  Tomcat-Requests_7441-"http-nio-8003"_Processin...                      6                                             4                       1615514400  trace.to_Tomcat01.duration_mean; trace.to_Tomc...
3        1615514400  Tomcat02               2      1.000833      1.000772  Tomcat-Requests_7441-"http-nio-8003"_Processin...                      6                                             2                       1615514400  trace.to_Tomcat02.duration_mean; trace.to_Tomc...
4        1615514400  Tomcat03               2      1.000805      1.000754  Tomcat-Requests_7441-"http-nio-8003"_Processin...                      6                                             1                       1615514400                     trace.to_Tomcat03.duration_p95
..              ...       ...             ...           ...           ...                                                ...                    ...                                           ...                              ...                                                ...
19       1615516140      IG01               1      1.000013      1.000013              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)                      0                                             0                             <NA>                                                   
20       1615516140      IG02               1      1.000013      1.000013              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)                      0                                             0                             <NA>                                                   
21       1615516140   Mysql02               1      1.000013      1.000013              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)                      0                                             0                             <NA>                                                   
22       1615516140  apache01               1      1.000013      1.000013              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)                      0                                             0                             <NA>                                                   
23       1615516140  apache02               1      1.000013      1.000013              OSLinux-OSLinux_ZABBIX_Host_Uptime(1)                      0                                             0                             <NA>                                                   

[24 rows x 10 columns],    start_timestamp          top_components had_trace_anomaly earliest_trace_anomaly_timestamp
0       1615514400  Redis01(9); Redis02(9)            no; no                           NA; NA
1       1615516140              Redis02(8)                no                               NA)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np

# Reuse kernel variables: sequences_df, trace_window_df, thresholds_trace_df, log_window_df, thresholds_log_df
# Target timestamps
target_starts = [1615514400, 1615516140]

# 1) Filter sequences to the two start timestamps
seq_target_df = sequences_df[sequences_df['start_timestamp'].isin(target_starts)].copy()

# Ensure numeric types
seq_target_df['severity'] = pd.to_numeric(seq_target_df['severity'], errors='coerce')
seq_target_df['length_in_points'] = pd.to_numeric(seq_target_df['length_in_points'], errors='coerce')
seq_target_df['start_timestamp'] = seq_target_df['start_timestamp'].astype(int)

# 2) Aggregate per (start_timestamp, cmdb_id): sequence_count, max_severity, avg_severity, top_kpis
grp = seq_target_df.groupby(['start_timestamp','cmdb_id'])
agg_seq = grp.agg(
    sequence_count = ('kpi_name','size'),
    max_severity   = ('severity', lambda s: float(np.nanmax(s.values)) if s.notna().any() else np.nan),
    avg_severity   = ('severity', lambda s: float(np.nanmean(s.values)) if s.notna().any() else np.nan)
).reset_index()

# top_kpis string
def top_kpis_str(s, topn=3):
    vc = s.value_counts().head(topn)
    return "; ".join([f"{k}({int(v)})" for k,v in vc.items()])

topk = grp['kpi_name'].apply(lambda s: top_kpis_str(s, topn=3)).reset_index().rename(columns={'kpi_name':'top_kpis'})
agg_seq = agg_seq.merge(topk, on=['start_timestamp','cmdb_id'], how='left')

# Helper to compute trace anomalies before-or-at start for a component
def analyze_traces_for_component(component, start_ts):
    pat = f"to_{component}"
    traces_rel = trace_window_df[trace_window_df['trace_name'].str.contains(pat, na=False)].copy()
    if traces_rel.empty:
        return 0, 0, pd.NA, ""
    # attach thresholds for trace series (match by cmdb_id and trace_name)
    traces_rel = traces_rel.merge(thresholds_trace_df, on=['cmdb_id','trace_name'], how='left')
    # mark anomaly points
    traces_rel['is_anom'] = ((traces_rel['value'] > traces_rel['p95']) | (traces_rel['value'] < traces_rel['p05']))
    # examine distinct trace_name series
    trace_series_examined = int(traces_rel['trace_name'].nunique())
    # anomalies at or before start
    anom_before = traces_rel[(traces_rel['is_anom']) & (traces_rel['timestamp'] <= start_ts)]
    if anom_before.empty:
        return trace_series_examined, 0, pd.NA, ""
    trace_series_with_anom = int(anom_before[['trace_name']].drop_duplicates().shape[0])
    earliest_trace_anom_ts = int(anom_before['timestamp'].min())
    examples = "; ".join(anom_before['trace_name'].drop_duplicates().head(5).tolist())
    return trace_series_examined, trace_series_with_anom, earliest_trace_anom_ts, examples

# Helper to compute log anomalies at-or-before start for a component
def analyze_logs_for_component(component, start_ts):
    logs_rel = log_window_df[log_window_df['cmdb_id'] == component].copy()
    if logs_rel.empty:
        return 0, 0, pd.NA
    logs_rel = logs_rel.merge(thresholds_log_df, on=['cmdb_id','log_name'], how='left')
    logs_rel['is_anom'] = ((logs_rel['value'] > logs_rel['p95']) | (logs_rel['value'] < logs_rel['p05']))
    log_series_examined = int(logs_rel['log_name'].nunique())
    anom_before = logs_rel[(logs_rel['is_anom']) & (logs_rel['timestamp'] <= start_ts)]
    if anom_before.empty:
        return log_series_examined, 0, pd.NA
    log_series_with_anom = int(anom_before[['log_name']].drop_duplicates().shape[0])
    earliest_log_anom_ts = int(anom_before['timestamp'].min())
    return log_series_examined, log_series_with_anom, earliest_log_anom_ts

# 3-5) For each aggregated row, compute trace and log related metrics and final_score
rows = []
for _, r in agg_seq.iterrows():
    ts = int(r['start_timestamp'])
    comp = r['cmdb_id']
    seq_count = int(r['sequence_count'])
    max_sev = float(r['max_severity']) if not pd.isna(r['max_severity']) else np.nan
    avg_sev = float(r['avg_severity']) if not pd.isna(r['avg_severity']) else np.nan
    top_kpis = r.get('top_kpis', "")

    # traces
    trace_examined, trace_with_anom, earliest_trace_ts, trace_examples = analyze_traces_for_component(comp, ts)
    # logs
    log_examined, log_with_anom, earliest_log_ts = analyze_logs_for_component(comp, ts)

    # final_score = sequence_count + 5*trace_series_with_anomaly_before_or_at_start + 3*log_series_with_anomaly_at_or_before_start + round((max_severity-1)*10)
    sev_component = 0 if pd.isna(max_sev) else int(round((max_sev - 1.0) * 10))
    final_score = seq_count + 5 * int(trace_with_anom) + 3 * int(log_with_anom) + int(sev_component)

    rows.append({
        'start_timestamp': ts,
        'cmdb_id': comp,
        'sequence_count': seq_count,
        'max_severity': max_sev,
        'avg_severity': avg_sev,
        'trace_series_examined': trace_examined,
        'trace_series_with_anomaly_before_or_at_start': int(trace_with_anom),
        'earliest_trace_anomaly_timestamp': earliest_trace_ts,
        'log_series_examined': log_examined,
        'log_series_with_anomaly_at_or_before_start': int(log_with_anom),
        'earliest_log_anomaly_timestamp': earliest_log_ts,
        'final_score': int(final_score)
    })

candidates_df = pd.DataFrame(rows)

# 6) Return top 5 components per timestamp sorted by final_score desc (limit 10 rows total)
candidates_df = candidates_df.sort_values(by=['start_timestamp','final_score'], ascending=[True,False])
top_candidates_df = candidates_df.groupby('start_timestamp').head(5).reset_index(drop=True)

# 7) Summary per timestamp: top-ranked component and its final_score
summary_rows = []
for ts in target_starts:
    df_ts = top_candidates_df[top_candidates_df['start_timestamp']==ts]
    if df_ts.empty:
        summary_rows.append({'start_timestamp': int(ts), 'top_component': pd.NA, 'top_final_score': pd.NA})
    else:
        top = df_ts.sort_values(by='final_score', ascending=False).iloc[0]
        summary_rows.append({'start_timestamp': int(ts), 'top_component': top['cmdb_id'], 'top_final_score': int(top['final_score'])})
summary_per_ts_df = pd.DataFrame(summary_rows)

# Final compact outputs
top_candidates_df, summary_per_ts_df

```
Out[10]:
```
```python
# Summarize the ranked candidate table for the two start timestamps in plain English.
summary = (
    "Ranked candidate summary for start timestamps 1615514400 and 1615516140 (compact):\n\n"
    "Timestamp 1615514400:\n"
    "- Top-ranked component: Tomcat01 (final_score 12). Tomcat02 is tied (final_score 12).\n"
    "- Other notable candidates: Redis01 and Redis02 (each sequence_count=9, final_score=9) and Tomcat03 (final_score=7).\n"
    "- Tomcat01 / Tomcat02 details: sequence_count=2, max_severity ≈1.0004 / 1.0008, trace_series_examined=3, "
    "trace_series_with_anomaly_before_or_at_start=2 (earliest related trace anomaly ts = 1615514400), "
    "log_series_examined=3 but log_series_with_anomaly_at_or_before_start=0 (no log anomalies at/before start).\n"
    "- Redis01/Redis02: high metric sequence counts (9) but no related trace or log anomalies detected at/before start.\n\n"
    "Timestamp 1615516140:\n"
    "- Top-ranked component: Tomcat01 (final_score 23). Tomcat02, Tomcat03, Tomcat04 also rank at similar top scores (23).\n"
    "- Redis02 is notable with sequence_count=8 and final_score=10 but no related trace/log anomalies.\n"
    "- Tomcat01–Tomcat04 details: sequence_count=2 each, max_severity ≈1.0017–1.0036, trace_series_examined=3, "
    "trace_series_with_anomaly_before_or_at_start=3 (earliest related trace anomaly ts often = 1615514400), "
    "log_series_examined=3 and log_series_with_anomaly_at_or_before_start=2 (earliest log anomaly timestamps vary per Tomcat, e.g., 1615514760/1615515120 etc).\n\n"
    "Overall observation:\n"
    "- Tomcat services are top ranked by final_score at both timestamps and show related trace anomalies occurring at or before the metric sequence starts, "
    "suggesting correlated trace-level anomalies prior to metric sequences.\n"
    "- Redis nodes show many metric sequences but no matching trace or log anomalies in the examined window—these may be isolated metric-level events or internal to Redis.\n\n"
    "Notes: final_score = sequence_count + 5*trace_anom_count + 3*log_anom_count + rounded((max_severity-1)*10). "
    "Timestamps are epoch seconds. Full per-row details are in kernel variable top_candidates_df and supporting tables."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   start_timestamp   cmdb_id  sequence_count  max_severity  avg_severity  trace_series_examined  trace_series_with_anomaly_before_or_at_start earliest_trace_anomaly_timestamp  log_series_examined  log_series_with_anomaly_at_or_before_start earliest_log_anomaly_timestamp  final_score
0       1615514400  Tomcat01               2      1.000367      1.000349                      3                                             2                       1615514400                    3                                           0                           <NA>           12
1       1615514400  Tomcat02               2      1.000833      1.000772                      3                                             2                       1615514400                    3                                           0                           <NA>           12
2       1615514400   Redis01               9      1.000083      1.000046                      0                                             0                             <NA>                    0                                           0                           <NA>            9
3       1615514400   Redis02               9      1.000076      1.000043                      0                                             0                             <NA>                    0                                           0                           <NA>            9
4       1615514400  Tomcat03               2      1.000805      1.000754                      3                                             1                       1615514400                    3                                           0                           <NA>            7
5       1615516140  Tomcat01               2      1.001727      1.001026                      3                                             3                       1615514400                    3                                           2                     1615514760           23
6       1615516140  Tomcat02               2      1.003534      1.002147                      3                                             3                       1615514400                    3                                           2                     1615515120           23
7       1615516140  Tomcat03               2      1.003646      1.002178                      3                                             3                       1615514400                    3                                           2                     1615514520           23
8       1615516140  Tomcat04               2      1.003427      1.002071                      3                                             3                       1615514400                    3                                           2                     1615514520           23
9       1615516140   Redis02               8      1.182749      1.022862                      0                                             0                             <NA>                    0                                           0                           <NA>           10,    start_timestamp top_component  top_final_score
0       1615514400      Tomcat01               12
1       1615516140      Tomcat01               23)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables: sequences_df, trace_window_df, thresholds_trace_df, log_window_df, thresholds_log_df
# Target components and timestamps
tomcats = ["Tomcat01","Tomcat02","Tomcat03","Tomcat04"]
target_starts = [1615514400, 1615516140]

# Step 1: sequences for these Tomcat components with start_timestamp in target_starts
step1_df = sequences_df[
    (sequences_df['cmdb_id'].isin(tomcats)) &
    (sequences_df['start_timestamp'].isin(target_starts))
].loc[:, ['cmdb_id','kpi_name','anomaly_type','start_timestamp','end_timestamp',
         'length_in_points','p95','p05','max_value_in_seq','min_value_in_seq','severity']].copy()
# Limit to 50 rows
step1_df = step1_df.head(50).reset_index(drop=True)

# Step 2: trace anomalies related to each Tomcat component (trace_name contains 'to_TomcatXX')
trace_anom_rows = []
for comp in tomcats:
    pat = f"to_{comp}"
    traces_rel = trace_window_df[trace_window_df['trace_name'].str.contains(pat, na=False)].copy()
    if traces_rel.empty:
        continue
    # attach thresholds for these trace rows
    traces_rel = traces_rel.merge(thresholds_trace_df, on=['cmdb_id','trace_name'], how='left', suffixes=('','_thr'))
    # mark anomaly type per row
    traces_rel['anomaly_type'] = np.where(traces_rel['value'] > traces_rel['p95'], 'high',
                                         np.where(traces_rel['value'] < traces_rel['p05'], 'low', pd.NA))
    # keep only anomaly rows
    traces_rel = traces_rel[traces_rel['anomaly_type'].notna()].copy()
    if traces_rel.empty:
        continue
    # For each target start timestamp, pick anomaly rows with timestamp <= start
    for st in target_starts:
        anom_before = traces_rel[traces_rel['timestamp'] <= st].copy()
        if anom_before.empty:
            continue
        # sort by timestamp ascending
        anom_before.sort_values('timestamp', inplace=True)
        # keep up to first 5 per (component, start)
        anom_sel = anom_before.head(5)
        for _, ar in anom_sel.iterrows():
            trace_anom_rows.append({
                'component': comp,
                'trace_name': ar['trace_name'],
                'anomaly_timestamp': int(ar['timestamp']),
                'value': float(ar['value']),
                'p95': float(ar['p95']) if not pd.isna(ar['p95']) else np.nan,
                'p05': float(ar['p05']) if not pd.isna(ar['p05']) else np.nan,
                'anomaly_type': ar['anomaly_type'],
                'start_timestamp': st
            })

# Combine and limit total to 20 rows
if trace_anom_rows:
    trace_anom_df = pd.DataFrame(trace_anom_rows)
    # sort by start_timestamp, component, anomaly_timestamp
    trace_anom_df = trace_anom_df.sort_values(by=['start_timestamp','component','anomaly_timestamp']).reset_index(drop=True)
    trace_anom_df_final = trace_anom_df.head(20).loc[:, ['component','trace_name','anomaly_timestamp','value','p95','p05','anomaly_type','start_timestamp']]
else:
    trace_anom_df_final = pd.DataFrame(columns=['component','trace_name','anomaly_timestamp','value','p95','p05','anomaly_type','start_timestamp'])

# Step 3: log anomalies per component and start_timestamp (use log_window_df and thresholds_log_df)
log_anom_rows = []
for comp in tomcats:
    logs_rel = log_window_df[log_window_df['cmdb_id'] == comp].copy()
    if logs_rel.empty:
        continue
    logs_rel = logs_rel.merge(thresholds_log_df, on=['cmdb_id','log_name'], how='left')
    logs_rel['anomaly_type'] = np.where(logs_rel['value'] > logs_rel['p95'], 'high',
                                       np.where(logs_rel['value'] < logs_rel['p05'], 'low', pd.NA))
    logs_rel = logs_rel[logs_rel['anomaly_type'].notna()].copy()
    if logs_rel.empty:
        continue
    for st in target_starts:
        anom_before = logs_rel[logs_rel['timestamp'] <= st].copy()
        if anom_before.empty:
            continue
        anom_before.sort_values('timestamp', inplace=True)
        anom_sel = anom_before.head(5)
        for _, ar in anom_sel.iterrows():
            log_anom_rows.append({
                'component': comp,
                'log_name': ar['log_name'],
                'anomaly_timestamp': int(ar['timestamp']),
                'value': float(ar['value']),
                'p95': float(ar['p95']) if not pd.isna(ar['p95']) else np.nan,
                'p05': float(ar['p05']) if not pd.isna(ar['p05']) else np.nan,
                'anomaly_type': ar['anomaly_type'],
                'start_timestamp': st
            })

# Combine and limit to 20 rows
if log_anom_rows:
    log_anom_df = pd.DataFrame(log_anom_rows)
    log_anom_df = log_anom_df.sort_values(by=['start_timestamp','component','anomaly_timestamp']).reset_index(drop=True)
    log_anom_df_final = log_anom_df.head(20).loc[:, ['component','log_name','anomaly_timestamp','value','p95','p05','anomaly_type','start_timestamp']]
else:
    log_anom_df_final = pd.DataFrame(columns=['component','log_name','anomaly_timestamp','value','p95','p05','anomaly_type','start_timestamp'])

# Step 4: Summary table per component per timestamp
summary_rows = []
for st in target_starts:
    for comp in tomcats:
        # sequence_count and earliest_sequence_start from step1_df
        seqs = step1_df[(step1_df['cmdb_id']==comp) & (step1_df['start_timestamp']==st)].copy()
        seq_count = int(seqs.shape[0])
        earliest_sequence_start = int(seqs['start_timestamp'].min()) if seq_count>0 else pd.NA
        # earliest related trace anomaly timestamp for this comp & start (from trace_anom_df_final)
        ta = trace_anom_df_final[(trace_anom_df_final['component']==comp) & (trace_anom_df_final['start_timestamp']==st)]
        earliest_trace_anom_ts = int(ta['anomaly_timestamp'].min()) if not ta.empty else pd.NA
        # earliest related log anomaly timestamp for this comp & start (from log_anom_df_final)
        la = log_anom_df_final[(log_anom_df_final['component']==comp) & (log_anom_df_final['start_timestamp']==st)]
        earliest_log_anom_ts = int(la['anomaly_timestamp'].min()) if not la.empty else pd.NA
        summary_rows.append({
            'start_timestamp': int(st),
            'cmdb_id': comp,
            'sequence_count': seq_count,
            'earliest_sequence_start': earliest_sequence_start,
            'earliest_related_trace_anomaly_timestamp': earliest_trace_anom_ts,
            'earliest_related_log_anomaly_timestamp': earliest_log_anom_ts
        })

summary_df = pd.DataFrame(summary_rows)
# Sort as requested
summary_df = summary_df.sort_values(by=['start_timestamp','sequence_count'], ascending=[True,False]).reset_index(drop=True)

# Return compact aggregates (variables)
step1_df, trace_anom_df_final, log_anom_df_final, summary_df

```
Out[11]:
```
# Compose a concise plain-English summary focused on Tomcat evidence using existing kernel tables
summary_text = (
    "Focused evidence for Tomcat01–Tomcat04 (timestamps 1615514400 and 1615516140):\n\n"
    "1) Metric sequences:\n"
    "- Each Tomcat (Tomcat01..Tomcat04) has 2 metric sequences starting at 1615514400 and 1615516140 (sequence_count=2 per component per timestamp).\n"
    "- Sequences at 1615514400 are metric 'low' anomalies (values < p05). Sequences at 1615516140 are metric 'high' anomalies (values > p95).\n"
    "- Severity of metric sequences is small but consistent (max severity ~1.0003–1.0036), i.e., values slightly above/below global thresholds.\n\n"
    "2) Trace evidence (correlation):\n"
    "- For all four Tomcats, trace anomalies on traces targeting the Tomcats (trace.to_TomcatXX.*) occur at or before the metric sequence starts.\n"
    "  * Many duration-based trace anomalies (duration_mean / duration_p95) show 'low' values at 1615514400.\n"
    "  * Several trace.row_count series show 'high' anomalies at timestamps before 1615516140 (examples include 1615514760, 1615514820, 1615515000 depending on Tomcat).\n"
    "- Earliest related trace anomaly reported for the Tomcats is 1615514400 (present for all Tomcats).\n\n"
    "3) Log evidence (correlation):\n"
    "- Log anomalies are more prominent before the 1615516140 sequences than before 1615514400.\n"
    "  * For 1615516140, Tomcat logs show gc_pause_max and log.row_count anomalies occurring prior to the metric sequence (examples: Tomcat01 log anomalies at 1615514760 and 1615515240; Tomcat02 log anomalies at 1615515120/1615515360/1615515240; Tomcat03/04 similar).\n"
    "  * For 1615514400, earliest related log anomaly is not present (NA) for these Tomcats.\n\n"
    "4) Conclusion and interpretation:\n"
    "- Tomcat01–Tomcat04 show temporally consistent signals across metrics, traces, and logs: trace anomalies (duration and row_count) occur at or before the metric sequence starts, and log anomalies (gc pause / row counts) appear before the later metric peak at 1615516140.\n"
    "- This temporal alignment (trace anomaly → metric sequence → log anomalies before the later peak) provides supporting evidence that Tomcat services were involved in the incident and are strong candidate components for investigation.\n"
    "- Note: metric severity values are small (>1 but near 1), so the absolute magnitude change is modest; the correlation across telemetry sources is the key signal.\n\n"
    "Data references: sequences (metric) in step1_df; related trace anomalies in trace_anom_df_final; related log anomalies in log_anom_df_final; per-component summary in summary_df."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name anomaly_type  start_timestamp  end_timestamp  length_in_points           p95           p05  max_value_in_seq  min_value_in_seq  severity
0   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  1.409071e+09  1.392857e+09      1.411505e+09      1.409738e+09  1.001727
1   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...         high       1615516140     1615516200                 2  2.182606e+06  2.167652e+06      2.183316e+06      2.182841e+06  1.000325
2   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  6.759160e+08  6.591909e+08      6.783048e+08      6.766749e+08  1.003534
3   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...         high       1615516140     1615516200                 2  9.802965e+05  9.654295e+05      9.810420e+05      9.805450e+05  1.000760
4   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  6.379476e+08  6.214076e+08      6.402738e+08      6.386512e+08  1.003646
5   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...         high       1615516140     1615516200                 2  9.786430e+05  9.637095e+05      9.793380e+05      9.788820e+05  1.000710
6   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...         high       1615516140     1615516200                 2  6.402499e+08  6.237299e+08      6.424442e+08      6.409380e+08  1.003427
7   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...         high       1615516140     1615516200                 2  9.792245e+05  9.640230e+05      9.799240e+05      9.794600e+05  1.000714
8   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...          low       1615514400     1615514460                 2  1.409071e+09  1.392857e+09      1.392667e+09      1.392395e+09  1.000332
9   Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  2.182606e+06  2.167652e+06      2.167378e+06      2.166856e+06  1.000367
10  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...          low       1615514400     1615514460                 2  6.759160e+08  6.591909e+08      6.590142e+08      6.587222e+08  1.000712
11  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.802965e+05  9.654295e+05      9.651780e+05      9.646260e+05  1.000833
12  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...          low       1615514400     1615514460                 2  6.379476e+08  6.214076e+08      6.212478e+08      6.209709e+08  1.000703
13  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.786430e+05  9.637095e+05      9.634800e+05      9.629340e+05  1.000805
14  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...          low       1615514400     1615514460                 2  6.402499e+08  6.237299e+08      6.235313e+08      6.232140e+08  1.000828
15  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...          low       1615514400     1615514460                 2  9.792245e+05  9.640230e+05      9.637660e+05      9.632030e+05  1.000851,    component                       trace_name  anomaly_timestamp       value         p95         p05 anomaly_type  start_timestamp
0   Tomcat01  trace.to_Tomcat01.duration_mean         1615514400    0.423525    3.215496    0.540828          low       1615514400
1   Tomcat01   trace.to_Tomcat01.duration_p95         1615514400    2.045500   11.280645    2.118437          low       1615514400
2   Tomcat01  trace.to_Tomcat01.duration_mean         1615514400    0.527463    3.130866    0.571107          low       1615514400
3   Tomcat01   trace.to_Tomcat01.duration_p95         1615514400    2.107400   10.637015    2.135442          low       1615514400
4   Tomcat02  trace.to_Tomcat02.duration_mean         1615514400    0.489387    3.003555    0.516542          low       1615514400
5   Tomcat02   trace.to_Tomcat02.duration_p95         1615514400    2.086800   11.058030    2.094563          low       1615514400
6   Tomcat03   trace.to_Tomcat03.duration_p95         1615514400    2.062000   11.421605    2.073385          low       1615514400
7   Tomcat04  trace.to_Tomcat04.duration_mean         1615514400    0.466386    2.845704    0.502602          low       1615514400
8   Tomcat01  trace.to_Tomcat01.duration_mean         1615514400    0.423525    3.215496    0.540828          low       1615516140
9   Tomcat01   trace.to_Tomcat01.duration_p95         1615514400    2.045500   11.280645    2.118437          low       1615516140
10  Tomcat01  trace.to_Tomcat01.duration_mean         1615514400    0.527463    3.130866    0.571107          low       1615516140
11  Tomcat01   trace.to_Tomcat01.duration_p95         1615514400    2.107400   10.637015    2.135442          low       1615516140
12  Tomcat01      trace.to_Tomcat01.row_count         1615514760  347.000000  340.250000  235.000000         high       1615516140
13  Tomcat02  trace.to_Tomcat02.duration_mean         1615514400    0.489387    3.003555    0.516542          low       1615516140
14  Tomcat02   trace.to_Tomcat02.duration_p95         1615514400    2.086800   11.058030    2.094563          low       1615516140
15  Tomcat02      trace.to_Tomcat02.row_count         1615514820  339.000000  329.450000  226.050000         high       1615516140
16  Tomcat02      trace.to_Tomcat02.row_count         1615515000  308.000000  306.650000  225.400000         high       1615516140
17  Tomcat02   trace.to_Tomcat02.duration_p95         1615515060    2.083400   11.058030    2.094563          low       1615516140
18  Tomcat03   trace.to_Tomcat03.duration_p95         1615514400    2.062000   11.421605    2.073385          low       1615516140
19  Tomcat03   trace.to_Tomcat03.duration_p95         1615514460    2.075000   12.672857    2.075338          low       1615516140,    component          log_name  anomaly_timestamp   value      p95      p05 anomaly_type  start_timestamp
0   Tomcat01  log.gc_pause_max         1615514760    0.09    0.078    0.020         high       1615516140
1   Tomcat01     log.row_count         1615514760  674.00  661.500  452.000         high       1615516140
2   Tomcat01  log.gc_pause_max         1615514820    0.08    0.078    0.020         high       1615516140
3   Tomcat01     log.row_count         1615515240  697.00  661.500  452.000         high       1615516140
4   Tomcat01     log.row_count         1615516020  447.00  661.500  452.000          low       1615516140
5   Tomcat02  log.gc_pause_max         1615515120    0.03    0.120    0.038          low       1615516140
6   Tomcat02  log.gc_pause_max         1615515240    0.03    0.120    0.038          low       1615516140
7   Tomcat02     log.row_count         1615515240  653.00  626.500  458.000         high       1615516140
8   Tomcat02  log.gc_pause_max         1615515360    0.14    0.120    0.038         high       1615516140
9   Tomcat02     log.row_count         1615515900  629.00  626.500  458.000         high       1615516140
10  Tomcat03     log.row_count         1615514520  464.00  634.500  466.500          low       1615516140
11  Tomcat03     log.row_count         1615514760  645.00  634.500  466.500         high       1615516140
12  Tomcat03  log.gc_pause_max         1615514820    0.09    0.080    0.020         high       1615516140
13  Tomcat03     log.row_count         1615515240  669.00  634.500  466.500         high       1615516140
14  Tomcat04  log.gc_pause_max         1615514520    0.11    0.107    0.023         high       1615516140
15  Tomcat04     log.row_count         1615514700  639.00  638.000  468.500         high       1615516140
16  Tomcat04  log.gc_pause_max         1615514760    0.02    0.107    0.023          low       1615516140
17  Tomcat04  log.gc_pause_max         1615515120    0.12    0.107    0.023         high       1615516140
18  Tomcat04     log.row_count         1615515180  641.00  638.000  468.500         high       1615516140,    start_timestamp   cmdb_id  sequence_count  earliest_sequence_start earliest_related_trace_anomaly_timestamp earliest_related_log_anomaly_timestamp
0       1615514400  Tomcat01               2               1615514400                               1615514400                                   <NA>
1       1615514400  Tomcat02               2               1615514400                               1615514400                                   <NA>
2       1615514400  Tomcat03               2               1615514400                               1615514400                                   <NA>
3       1615514400  Tomcat04               2               1615514400                               1615514400                                   <NA>
4       1615516140  Tomcat01               2               1615516140                               1615514400                             1615514760
5       1615516140  Tomcat02               2               1615516140                               1615514400                             1615515120
6       1615516140  Tomcat03               2               1615516140                               1615514400                             1615514520
7       1615516140  Tomcat04               2               1615516140                                     <NA>                             1615514520)```
```